# WARP for Encoder-only Models — DistilBERT / BERT on SST-2

1. Run §1–§9 top to bottom (defines everything, doesn't start experiments).
2. Phase 1 → Phase 2 is the main WARP experiment: load + pooler FT, build caches, run the QP repair, then the full certificate evaluation.
3. Phase 3 is the ablation sweep — optional.
4. The Baselines section (LoRA / Full FT) rebuilds its own raw-text datasets and only needs §1–§6; it does not depend on Phases 1–3.


### Before running

`dev_ann.json` and `test_ann.json` (AdvGLUE) must already be in the Drive folder named by `DRIVE_SUBDIR` (default `WARP Stuff`). The check cell in §2 reports what is present or missing — it never downloads or creates files.

To run against your own Drive folder or a local path, edit `DRIVE_SUBDIR` in §2; no other cell needs changing.


## §1. Install & imports

In [ ]:
# Install
!pip install transformers datasets scipy sentencepiece osqp -q
# This notebook is pure text classification (DistilBERT/BERT) and never touches
# torchvision. Recent `datasets` versions, once torchvision is importable, probe
# torchvision.io.VideoReader for a type check; recent torchvision has
# deprecated/removed VideoReader, so that probe raises — even though none of our
# code uses any video feature. Removing torchvision keeps the probe from firing.
!pip uninstall -y torchvision -q
print('Done.')

In [ ]:
# Imports
import json, time, warnings, os, pickle, types, copy
import numpy as np
import torch
import torch.nn as nn
import scipy.sparse as sp
import osqp
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict
from torch.utils.data import DataLoader, Dataset  # used by the Baselines section below
from scipy.sparse.linalg import svds
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, TrainerCallback
from transformers.modeling_outputs import SequenceClassifierOutput

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
rng    = np.random.default_rng(SEED)
print(f'Device: {DEVICE}')
import transformers, datasets as datasets_pkg, scipy, platform
ENV_FINGERPRINT = {
    'python':       platform.python_version(),
    'torch':        torch.__version__,
    'transformers': transformers.__version__,
    'datasets':     datasets_pkg.__version__,
    'numpy':        np.__version__,
    'scipy':        scipy.__version__,
    'osqp':         getattr(osqp, '__version__', 'unknown'),
    'gpu':          (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'),
}
for _k, _v in ENV_FINGERPRINT.items():
    print(f'  {_k:<13}= {_v}')


## §2. Paths & prerequisite check

In [ ]:
# Mount Drive
# Configurable: folder holding the data and the results. To run against your own
# Drive folder or a local path, edit this line only — no other cell changes.
DRIVE_SUBDIR = 'WARP Stuff'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_SUBDIR}'
except Exception:
    # Not on Colab (local / other notebook service): fall back to a relative path
    DRIVE_DIR = f'./{DRIVE_SUBDIR}'
    print(f'[info] not on Colab or mount failed, using local path: {DRIVE_DIR}')

os.makedirs(DRIVE_DIR, exist_ok=True)
DEV_PATH   = f'{DRIVE_DIR}/dev_ann.json'
TEST_PATH  = f'{DRIVE_DIR}/test_ann.json'
RESULT_DIR = f'{DRIVE_DIR}/results_full'
os.makedirs(RESULT_DIR, exist_ok=True)

# Baseline (LoRA / Full FT) results go to their own folder so they cannot
# overwrite the WARP results above.
BASELINE_RESULT_DIR = f'{DRIVE_DIR}/results_baseline'
os.makedirs(BASELINE_RESULT_DIR, exist_ok=True)

CACHE_PATHS = {
    'distilbert': f'{DRIVE_DIR}/distilbert_embeddings.pkl',
    'bert':       f'{DRIVE_DIR}/bert_poolerft_embeddings.pkl',
}
print(f'Data dir   -> {DRIVE_DIR}')
print(f'Results -> {RESULT_DIR}')


In [ ]:
# Check: confirm the prerequisite files are in place — never downloads or creates
import os

_checks = {
    'dev_ann.json  (source of the repair set)':          DEV_PATH,
    'test_ann.json (source of attack generalization)':   TEST_PATH,
}

print(f'DRIVE_DIR = {DRIVE_DIR}\n')
_all_ok = True
for _desc, _path in _checks.items():
    _ok = os.path.exists(_path)
    _all_ok = _all_ok and _ok
    print(f"{'OK     ' if _ok else 'MISSING'} {_desc}")
    print(f"    {_path}")

print()
if _all_ok:
    print('All required files present — continue running the cells below.')
else:
    print('Some files are missing. Follow the README "Before running" section and\n'
          '   upload the files marked MISSING above to the paths shown (normally\n'
          f'   dev_ann.json / test_ann.json into the Drive folder "{DRIVE_DIR}"),\n'
          '   then continue.')


## §3. Global config & hyperparameters

In [ ]:
# Global hyperparameters
LAMBDA    = 50.0
MARGIN_H  = 0.3
MARGIN_S  = 1.0
MAX_ITER  = 300
N_REMAIN  = 800
RANK      = 2
N_PERTURB = 200
REG       = 2.0      # QP regularization (same for all models)

# FT config for pooler-based models
FT_LR         = 1e-3
FT_BATCH_SIZE = 32
FT_MAX_STEPS  = 30

# Ablation sweep values
# NOTE: repair_size last value is set dynamically per model (= full repair set size)
ABLATION_REPAIR_SIZES_BASE = [10, 25, 50, 75]   # last value appended per-arch at runtime
ABLATION_REMAIN_SIZES      = [0, 50, 100, 200, 400, 800, 1200, 1600, 2400, 3200]
ABLATION_REG_VALUES        = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
ABLATION_LAMBDA_VALUES     = [10.0, 25.0, 50.0, 100.0, 200.0, 500.0]
ABLATION_MARGIN_S_VALUES   = [0.3, 0.5, 1.0, 2.0, 5.0]
ABLATION_MARGIN_H_VALUES   = [0.05, 0.1, 0.3, 0.5, 1.0]
ABLATION_RANK_VALUES = [1, 2, 4, 8, 16, 32]

# Excluded attack methods (too few samples to be meaningful)
EXCLUDED_METHODS = {'AdvFever', 'BERT-ATTACK', 'SCPN', 'SemAttack'}

FIXED_V0_STORE = {}  # per-arch fixed v0 for reproducibility

print(f'REG={REG}  LAMBDA={LAMBDA}  RANK={RANK}  MARGIN_S={MARGIN_S}')
print(f'FT_MAX_STEPS={FT_MAX_STEPS}  FT_LR={FT_LR}')
print(f'Excluded attack methods: {EXCLUDED_METHODS}')

# Baseline (LoRA / Full FT) hyperparameters
BL_LR         = 1e-4
BL_BATCH_SIZE = 16
BL_EPOCHS     = 300
BL_LORA_RANK  = 2      # same as QP RANK
BL_LORA_ALPHA = 4
BL_PATIENCE   = 5

print(f'[baseline] LR={BL_LR}  epochs={BL_EPOCHS}  lora_rank={BL_LORA_RANK}  patience={BL_PATIENCE}')

## §4. Architecture layer: activations, model configs, prediction utilities

In [ ]:
# Activation functions
def act_relu(x):  return np.maximum(0.0, x)
def dact_relu(x): return (x > 0).astype(np.float64)

def act_tanh(x):  return np.tanh(x)
def dact_tanh(x): return 1.0 - np.tanh(x)**2

def act_gelu(x):
    c = np.sqrt(2.0 / np.pi)
    return 0.5 * x * (1.0 + np.tanh(c * (x + 0.044715 * x**3)))

def dact_gelu(x):
    c   = np.sqrt(2.0 / np.pi)
    arg = c * (x + 0.044715 * x**3)
    t   = np.tanh(arg)
    s2  = 1.0 - t**2
    da  = c * (1.0 + 3.0 * 0.044715 * x**2)
    return 0.5 * (1.0 + t) + 0.5 * x * s2 * da

# Lipschitz constant of activation
ACT_LIP  = {'distilbert': 1.0, 'bert': 1.0}
ACT_FN   = {'distilbert': act_relu, 'bert': act_tanh}
DACT_FN  = {'distilbert': dact_relu, 'bert': dact_tanh}

xs = np.linspace(-5, 5, 10000)
print(f"max |GELU'(x)| = {np.abs(dact_gelu(xs)).max():.4f}  (expect ~1.13)")
print('Activations defined.')

In [ ]:
# Model configs
# hook_fn(model)   -> the layer whose INPUT we hook (= repair layer input = v)
# cls_fn(model)    -> the final linear classification layer (Wc)
# freeze_fn(model) -> list of params to FREEZE during FT
# ft_fn(model)     -> list of params to TRAIN during FT (None if no FT)

# Add or swap models here
MODEL_CONFIGS = {
    'distilbert': {
        'model_id':  'distilbert-base-uncased-finetuned-sst-2-english',
        'needs_ft':  True,
        'hook_fn':   lambda m: m.pre_classifier,
        'cls_fn':    lambda m: m.classifier,
        'freeze_fn': lambda m: list(m.distilbert.parameters()),
        'ft_fn':     lambda m: (list(m.pre_classifier.parameters()) +
                                list(m.classifier.parameters())),
        # ↓ the two fields below are used by the Baselines section only (LoRA / Full FT)
        'samelayer_unfreeze': lambda m: (list(m.pre_classifier.parameters()) +
                                        list(m.classifier.parameters())),
        'lora_target':        'pre_classifier',
    },
    'bert': {
        'model_id':  'textattack/bert-base-uncased-SST-2',
        'needs_ft':  True,
        'hook_fn':   lambda m: m.bert.pooler.dense,
        'cls_fn':    lambda m: m.classifier,
        'freeze_fn': lambda m: (list(m.bert.encoder.parameters()) +
                                list(m.bert.embeddings.parameters())),
        'ft_fn':     lambda m: (list(m.bert.pooler.parameters()) +
                                list(m.classifier.parameters())),
        'samelayer_unfreeze': lambda m: (list(m.bert.pooler.parameters()) +
                                        list(m.classifier.parameters())),
        'lora_target':        'bert.pooler.dense',
    },
    # Baselines-only config: the QP path needs hook_fn / cls_fn / needs_ft, which
    # are not provided here, so Phase 1-3 never touch this key. Paper Table 3
    # reports no RoBERTa numbers.
    'roberta': {
        'model_id':           'textattack/roberta-base-SST-2',
        'samelayer_unfreeze': lambda m: list(m.classifier.parameters()),
        'lora_target':        'classifier.dense',
    },
}

print('Model configs:')
for k, v in MODEL_CONFIGS.items():
    if 'needs_ft' in v:
        print(f'  {k:<16} needs_ft={v["needs_ft"]}  {v["model_id"]}')
    else:
        print(f'  {k:<16} (baseline-only config, no QP/needs_ft fields)  {v["model_id"]}')

In [ ]:
# Prediction utilities
def forward_w(v, W_dict):
    arch = W_dict['arch']
    pre  = W_dict['W'] @ v + W_dict['b']
    h    = ACT_FN[arch](pre)
    sc   = W_dict['Wc'] @ h + W_dict['bc']
    return sc

def predict_w(v, W_dict):
    return int(np.argmax(forward_w(v, W_dict)))

def gap_w(v, tgt, W_dict):
    sc = forward_w(v, W_dict)
    return float(sc[tgt] - sc[1 - tgt])

def eval_acc(cache, W_dict, use_gt=True):
    key = 'gt' if use_gt else 'pred'
    if not cache: return 0.0
    return 100.0 * sum(predict_w(s['v'], W_dict) == s[key]
                       for s in cache) / len(cache)

print('Prediction utilities defined.')

## §5. Model prep: pooler FT with CoeffNorm (GSN) tracking

In [ ]:
# Model loader + FT
LOADED_MODELS = {}   # arch -> {model, tok, W0, b0, Wc0, bc0, infer_fn,
                     #          W0_orig, b0_orig, Wc0_orig, bc0_orig, ...}
                     # W0/b0/Wc0/bc0      = weights AFTER FT (used by QP)
                     # W0_orig/...        = weights BEFORE FT (for 3-stage eval)
FT_COEFF_LOGS = {}   # arch -> list of {step, coeff_norm, spec_norm}
FT_BEST_STEPS = {}   # arch -> best step index

FT_TEST_SENTS = [
    "this movie is terrible and boring",
    "i really enjoyed this film",
    "the acting was awful",
    "a wonderful and moving story",
    "complete waste of time",
    "absolutely brilliant performance",
    "i fell asleep halfway through",
    "the best film i have seen this year",
]

def compute_coeff_norm_generic(model, hook_layer, cls_layer, arch, sents, tok):
    """Compute coefficient norm on a list of sentences."""
    model.eval()
    W  = hook_layer.weight.detach().cpu().numpy().astype(np.float64).copy()
    b  = hook_layer.bias.detach().cpu().numpy().astype(np.float64).copy()
    Wc = cls_layer.weight.detach().cpu().numpy().astype(np.float64).copy()
    bc = cls_layer.bias.detach().cpu().numpy().astype(np.float64).copy()

    vecs = []
    def hfn(m, inp, out):
        vecs.append(inp[0][0].detach().cpu().numpy().astype(np.float64))
    h = hook_layer.register_forward_hook(hfn)
    for sent in sents:
        enc = tok(sent, return_tensors='pt', truncation=True, max_length=128)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            model(**enc)
    h.remove()
    model.train()

    d_out, d_in = W.shape
    v0 = np.random.default_rng(42).standard_normal(min(d_out, d_in))
    v0 /= np.linalg.norm(v0)
    U, _, _ = svds(W, k=2, v0=v0)
    cols = np.argsort(np.var(np.stack(vecs), axis=0))[::-1][:d_in].copy()

    norms = []
    for v in vecs:
        pre   = W @ v + b
        jac   = DACT_FN[arch](pre)
        h_v   = ACT_FN[arch](pre)
        sc    = Wc @ h_v + bc
        tgt   = int(np.argmax(sc))
        wdif  = Wc[tgt] - Wc[1-tgt]
        coeffs = np.zeros((2, len(cols)))
        for r in range(2):
            eff_r = float((wdif * jac * U[:, r]).sum())
            coeffs[r] = eff_r * v[cols]
        norms.append(float(np.linalg.norm(coeffs)))

    spec = float(np.linalg.norm(W, ord=2))
    return float(np.mean(norms)), spec


def load_and_ft_model(arch):
    """
    Load model. If needs_ft, run pooler FT with per-step coefficient norm
    tracking, save best checkpoint.

    Stores in LOADED_MODELS[arch]:
      W0_orig / b0_orig / Wc0_orig / bc0_orig  — weights BEFORE FT
      W0 / b0 / Wc0 / bc0                      — weights AFTER FT (or same if no FT)
      infer_fn                                  — always uses POST-FT model
    """
    cfg      = MODEL_CONFIGS[arch]
    model_id = cfg['model_id']
    print(f'\n=== {arch} ({model_id}) ===')

    tok   = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id)
    model = model.to(DEVICE)
    model.eval()

    l2id   = {k.lower(): v for k, v in model.config.label2id.items()}
    POS_ID = l2id.get('positive', 1)

    hook_layer = cfg['hook_fn'](model)
    cls_layer  = cfg['cls_fn'](model)
    print(f'  hook: {hook_layer.__class__.__name__} {tuple(hook_layer.weight.shape)}')
    print(f'  cls:  {cls_layer.__class__.__name__}  {tuple(cls_layer.weight.shape)}')

    # Capture original (pre-FT) weights
    W0_orig  = hook_layer.weight.detach().cpu().numpy().astype(np.float64).copy()
    b0_orig  = hook_layer.bias.detach().cpu().numpy().astype(np.float64).copy()
    Wc0_orig = cls_layer.weight.detach().cpu().numpy().astype(np.float64).copy()
    bc0_orig = cls_layer.bias.detach().cpu().numpy().astype(np.float64).copy()

    # Coefficient norm at step 0
    cn0, spec0 = compute_coeff_norm_generic(
        model, hook_layer, cls_layer, arch, FT_TEST_SENTS, tok)
    print(f'  CoeffNorm step=0: {cn0:.4f}  ||W||_2={spec0:.4f}')

    coeff_log = [{'step': 0, 'coeff_norm': cn0, 'spec_norm': spec0}]
    best       = {'cn': cn0, 'step': 0,
                  'state': {k: v.cpu().clone() for k, v in model.state_dict().items()}}

    if cfg['needs_ft']:
        print(f'  Pooler FT: tracking CoeffNorm every step up to {FT_MAX_STEPS}...')

        # Freeze/unfreeze
        for p in model.parameters():      p.requires_grad = False
        for p in cfg['ft_fn'](model):     p.requires_grad = True
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Trainable params: {trainable:,}')

        # Data
        sst2 = load_dataset('nyu-mll/glue', 'sst2')
        def tok_fn(batch):
            return tok(batch['sentence'], truncation=True,
                       max_length=128, padding='max_length')
        sst2_tok = sst2.map(tok_fn, batched=True)
        sst2_tok = sst2_tok.rename_column('label', 'labels')
        cols_needed = ['input_ids', 'attention_mask', 'labels']
        if 'token_type_ids' in sst2_tok['train'].column_names:
            cols_needed.append('token_type_ids')
        sst2_tok.set_format('torch', columns=cols_needed)

        _refs = {'model': model, 'hook': hook_layer, 'cls': cls_layer,
                 'arch': arch, 'tok': tok, 'log': coeff_log, 'best': best}

        class StepCB(TrainerCallback):
            def on_step_end(self, args, state, control, **kwargs):
                cn, spec = compute_coeff_norm_generic(
                    _refs['model'], _refs['hook'], _refs['cls'],
                    _refs['arch'], FT_TEST_SENTS, _refs['tok'])
                entry = {'step': state.global_step,
                         'coeff_norm': cn, 'spec_norm': spec}
                _refs['log'].append(entry)
                print(f'  step={state.global_step:5d}  '
                      f'CoeffNorm={cn:.4f}  ||W||_2={spec:.4f}')
                if cn > _refs['best']['cn']:
                    _refs['best']['cn']    = cn
                    _refs['best']['step']  = state.global_step
                    _refs['best']['state'] = {
                        k: v.cpu().clone()
                        for k, v in _refs['model'].state_dict().items()}
                if state.global_step >= FT_MAX_STEPS:
                    control.should_training_stop = True

        ft_args = TrainingArguments(
            output_dir=f'/content/ft_{arch}',
            num_train_epochs=100,
            per_device_train_batch_size=FT_BATCH_SIZE,
            learning_rate=FT_LR,
            eval_strategy='no',
            save_strategy='no',
            logging_steps=100,
            report_to='none',
        )
        trainer = Trainer(
            model=model,
            args=ft_args,
            train_dataset=sst2_tok['train'],
            callbacks=[StepCB()],
        )
        model.train()
        trainer.train()

        print(f'\n  Best: CoeffNorm={best["cn"]:.4f} at step={best["step"]}')
        model.load_state_dict(best['state'])
        model = model.to(DEVICE)
        model.eval()
        for p in model.parameters(): p.requires_grad = False

        FT_COEFF_LOGS[arch] = coeff_log
        FT_BEST_STEPS[arch] = best['step']

    # Extract post-FT weights
    W0  = hook_layer.weight.detach().cpu().numpy().astype(np.float64).copy()
    b0  = hook_layer.bias.detach().cpu().numpy().astype(np.float64).copy()
    Wc0 = cls_layer.weight.detach().cpu().numpy().astype(np.float64).copy()
    bc0 = cls_layer.bias.detach().cpu().numpy().astype(np.float64).copy()

    cn_f, spec_f = compute_coeff_norm_generic(
        model, hook_layer, cls_layer, arch, FT_TEST_SENTS, tok)
    print(f'  CoeffNorm (final): {cn_f:.4f}  ||W||_2={spec_f:.4f}')

    # Build inference function (POST-FT model)
    def infer_fn(text):
        cap = {}
        def hk(m, inp, out):
            cap['v'] = inp[0][0].detach().cpu().numpy().astype(np.float64)
        h = hook_layer.register_forward_hook(hk)
        enc = tok(text, return_tensors='pt', truncation=True, max_length=128)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        h.remove()
        pred = 1 if int(out.logits[0].argmax()) == POS_ID else 0
        return cap['v'], pred

    LOADED_MODELS[arch] = {
        'model': model, 'tok': tok, 'infer_fn': infer_fn,
        # Post-FT weights (used for QP)
        'W0': W0, 'b0': b0, 'Wc0': Wc0, 'bc0': bc0,
        # Pre-FT weights (used for 3-stage eval; same as post-FT if needs_ft=False)
        'W0_orig': W0_orig, 'b0_orig': b0_orig,
        'Wc0_orig': Wc0_orig, 'bc0_orig': bc0_orig,
        'POS_ID': POS_ID, 'coeff_norm_final': cn_f,
    }
    print(f'  {arch} ready.')

print('Model loader defined.')

## §6. Cache builder

**Set membership is decided by the NumPy float64 forward pass** (`predict_w`), never by PyTorch float32 inference. The two can disagree for samples sitting near the decision boundary; if PyTorch predictions were used to filter, some remain/clean samples would end up with `s['pred'] != predict_w(v, W0)` and the before-QP accuracies would not be exactly 100%. Filtering with `predict_w` guarantees:

| set | membership condition | before-QP accuracy |
|---|---|---|
| `repair_set` | `predict_w(v, W0) != gt` | 0% |
| `remain_set` | `predict_w(v, W0) == np_pred` | 100% |
| `clean_set` | `predict_w(v, W0) == np_pred` | 100% |

For `needs_ft` models the cache is always rebuilt rather than loaded from disk, so that every `v` vector and predicted label reflects the post-FT weights.


In [ ]:
# Cache helpers (set membership uses the NumPy float64 forward — see §6)

DATASETS = {}  # arch -> {repair_set, remain_set, test_caches, clean_set}

N_CLEAN_TARGET = 2000
N_CLEAN_POOL   = int(N_CLEAN_TARGET * 2.5)


def _np_pred(v, W0, b0, Wc0, bc0, arch):
    """NumPy float64 forward pass — used for set-membership decisions."""
    pre = W0 @ v + b0
    h   = ACT_FN[arch](pre)
    sc  = Wc0 @ h + bc0
    return int(np.argmax(sc))


def build_cache(arch):
    infer_fn = LOADED_MODELS[arch]['infer_fn']
    m        = LOADED_MODELS[arch]
    W0, b0, Wc0, bc0 = m['W0'], m['b0'], m['Wc0'], m['bc0']
    rng_     = np.random.default_rng(SEED)

    with open(DEV_PATH)  as f: dev_ann  = json.load(f)
    with open(TEST_PATH) as f: test_ann = json.load(f)

    dev_sst2  = dev_ann['sst2']

    # Filter out excluded attack methods
    test_sst2 = [x for x in test_ann['sst2']
                 if x.get('method') not in EXCLUDED_METHODS
                 and x.get('method') != 'glue']
    BY_METHOD = defaultdict(list)
    for item in test_sst2:
        BY_METHOD[item['method']].append(item)

    def cache_items(items, key_text='sentence', key_label='label', desc=''):
        out = []
        for i, item in enumerate(items):
            gt      = int(item[key_label])
            v, _    = infer_fn(item[key_text])   # _ = torch pred (not used for filtering)
            np_p    = _np_pred(v, W0, b0, Wc0, bc0, arch)
            out.append({'v': v, 'gt': gt, 'pred': np_p, 'correct': np_p == gt,
                        'method': item.get('method', 'clean')})
            if (i+1) % 100 == 0:
                print(f'  {desc} {i+1}/{len(items)}')
        print(f'  {desc} done ({len(out)})')
        return out

    print(f'[{arch}] Caching dev (post-FT model, NumPy preds)...')
    dev_cache  = cache_items(dev_sst2, desc='dev')
    # repair_set = samples where NumPy forward gives WRONG answer
    repair_set = [s for s in dev_cache if not s['correct']]
    print(f'  repair_set: {len(repair_set)}')

    print(f'[{arch}] Caching test attacks (excluding: {EXCLUDED_METHODS})...')
    test_caches = {m: cache_items(items, desc=m)
                   for m, items in BY_METHOD.items()}
    print(f'  attack methods kept: {sorted(test_caches.keys())}')

    print(f'[{arch}] Caching Yelp (remain + clean)...')
    yelp = load_dataset('fancyzhx/yelp_polarity', split='test')
    pool = [{'text': ex['text'][:256], 'label': int(ex['label'])} for ex in yelp]
    rng_.shuffle(pool)

    # Remain set: first N_REMAIN*2 pool entries
    # Keep only entries where NumPy forward is CORRECT (consistent with QP).
    remain_all = []
    for i, item in enumerate(pool[:N_REMAIN * 2]):
        v, _  = infer_fn(item['text'])
        np_p  = _np_pred(v, W0, b0, Wc0, bc0, arch)
        remain_all.append({'v': v, 'gt': item['label'], 'pred': np_p,
                            'correct': np_p == item['label']})
        if (i+1) % 200 == 0: print(f'  yelp remain {i+1}')
    remain_set = [s for s in remain_all if s['correct']][:N_REMAIN]

    # Clean set: next N_CLEAN_POOL entries (non-overlapping)
    # Keep only entries where NumPy forward is CORRECT.
    clean_start = N_REMAIN * 2
    clean_all = []
    for item in pool[clean_start: clean_start + N_CLEAN_POOL]:
        v, _  = infer_fn(item['text'])
        np_p  = _np_pred(v, W0, b0, Wc0, bc0, arch)
        clean_all.append({'v': v, 'gt': item['label'], 'pred': np_p,
                           'correct': np_p == item['label']})
    clean_set = [s for s in clean_all if s['correct']][:N_CLEAN_TARGET]

    print(f'  remain={len(remain_set)}  clean={len(clean_set)} (target={N_CLEAN_TARGET})')
    if len(clean_set) < N_CLEAN_TARGET:
        print(f'  WARNING: only got {len(clean_set)} clean samples (wanted {N_CLEAN_TARGET})')

    return {'repair_set': repair_set, 'remain_set': remain_set,
            'test_caches': test_caches, 'clean_set': clean_set}


def load_or_build_cache(arch):
    """
    For needs_ft=True models: ALWAYS rebuild (post-FT v vectors required).
    For needs_ft=False models: load from disk if valid, else rebuild.
    In both cases the cache uses NumPy preds for set membership.
    """
    cfg  = MODEL_CONFIGS[arch]
    path = CACHE_PATHS[arch]

    if cfg['needs_ft']:
        print(f'[{arch}] needs_ft=True -> always rebuilding cache with post-FT model')
        d = build_cache(arch)
        with open(path, 'wb') as f: pickle.dump(d, f)
        print(f'[{arch}] Cache saved: {path}')
        return d

    # needs_ft=False: safe to load from disk if valid
    if os.path.exists(path):
        print(f'[{arch}] Loading cache: {path}')
        with open(path, 'rb') as f:
            d = pickle.load(f)

        needs_rebuild = False
        if 'clean_set' not in d:
            print('  Missing clean_set -> rebuilding')
            needs_rebuild = True
        elif len(d.get('clean_set', [])) < N_CLEAN_TARGET:
            print(f'  clean_set too small ({len(d["clean_set"])} < {N_CLEAN_TARGET}) -> rebuilding')
            needs_rebuild = True
        elif any(m in d.get('test_caches', {}) for m in EXCLUDED_METHODS):
            print(f'  Excluded methods present in cache -> rebuilding')
            needs_rebuild = True

        if needs_rebuild:
            d = build_cache(arch)
            with open(path, 'wb') as f: pickle.dump(d, f)
            print(f'[{arch}] Cache rebuilt and saved.')
        else:
            print(f'  repair={len(d["repair_set"])}  '
                  f'remain={len(d["remain_set"])}  '
                  f'clean={len(d["clean_set"])}  '
                  f'attacks={sorted(d["test_caches"].keys())}')
        return d

    print(f'[{arch}] Building cache...')
    d = build_cache(arch)
    with open(path, 'wb') as f: pickle.dump(d, f)
    print(f'[{arch}] Saved: {path}')
    return d

print('Cache helpers defined. (set membership uses NumPy float64 forward)')

## §7. QP repair

In [ ]:
# QP solver
def build_basis(W_np, rank, repair_cache, arch):
    if rank == 'full' or rank is None:
        d_out, d_in = W_np.shape
        U    = np.eye(d_out)
        cols = np.arange(d_in)
        return U, cols
    d_out, d_in = W_np.shape
    rank_ = min(rank, min(d_out, d_in) - 1)
    v0 = FIXED_V0_STORE.get(arch)
    if v0 is not None: v0 = v0[:min(d_out, d_in)]
    U, _, _ = svds(W_np, k=rank_, v0=v0)
    U = U.copy()
    Vrep = np.stack([s['v'] for s in repair_cache])
    cols = np.argsort(np.var(Vrep, axis=0))[::-1][:d_in].copy()
    return U, cols


def gap_and_coeffs(v, tgt, W_dict, U, cols):
    arch  = W_dict['arch']
    other = 1 - tgt
    rank  = U.shape[1]
    K     = len(cols)
    pre   = W_dict['W'] @ v + W_dict['b']
    jac   = DACT_FN[arch](pre)
    h     = ACT_FN[arch](pre)
    sc    = W_dict['Wc'] @ h + W_dict['bc']
    gap   = float(sc[tgt] - sc[other])
    wdif  = W_dict['Wc'][tgt] - W_dict['Wc'][other]
    coeffs = np.zeros((rank, K))
    for r in range(rank):
        eff_r     = float((wdif * jac * U[:, r]).sum())
        coeffs[r] = eff_r * v[cols]
    return gap, coeffs


def solve_qp(repair_cache, remain_cache, W_dict, U, cols,
             lambda_val=LAMBDA, reg_val=REG):
    rank = U.shape[1]
    K    = len(cols)
    n_s  = len(repair_cache)
    n_p  = len(remain_cache)
    nx   = rank * K + n_s
    nc   = n_s + n_p + n_s

    A = np.zeros((nc, nx))
    l = np.empty(nc)
    u = np.full(nc, 1e10)

    for i, s in enumerate(repair_cache):
        g, co = gap_and_coeffs(s['v'], s['gt'], W_dict, U, cols)
        for r in range(rank): A[i, r*K:(r+1)*K] = co[r]
        A[i, rank*K + i] = 1.0
        l[i] = MARGIN_S - g

    for j, s in enumerate(remain_cache):
        g, co = gap_and_coeffs(s['v'], s['pred'], W_dict, U, cols)
        for r in range(rank): A[n_s+j, r*K:(r+1)*K] = co[r]
        l[n_s+j] = MARGIN_H - g

    for i in range(n_s):
        A[n_s+n_p+i, rank*K+i] = 1.0
        l[n_s+n_p+i] = 0.0

    H = np.zeros((nx, nx))
    for r in range(rank):
        s_, e_ = r*K, (r+1)*K
        H[s_:e_, s_:e_] = reg_val * np.eye(K)
    g_vec = np.zeros(nx)
    g_vec[rank*K:] = lambda_val

    try:
        prob = osqp.OSQP()
        prob.setup(sp.csc_matrix(H), g_vec, sp.csc_matrix(A), l, u,
                   warm_starting=True, verbose=False,
                   max_iter=8000, eps_abs=1e-5, eps_rel=1e-5,
                   adaptive_rho=True, polish=True)
        res = prob.solve()
        if res.info.status not in ('solved', 'solved_inaccurate'):
            return None
        return res.x[:rank * K]
    except Exception as e:
        print(f'  QP exception: {e}')
        return None


def apply_alpha(alpha, W_dict, U, cols):
    rank = U.shape[1]
    K    = len(cols)
    dW   = np.zeros_like(W_dict['W'])
    for r in range(rank):
        dW[:, cols] += np.outer(U[:, r], alpha[r*K:(r+1)*K])
    W_dict['W'] = W_dict['W'] + dW
    return float(np.linalg.norm(dW, 'fro'))

print('QP solver defined.')

In [ ]:
# Repair runner
GAP_TOL = 1e-6

def run_repair(arch, repair_set, remain_set,
               rank=RANK, max_iter=MAX_ITER,
               lambda_val=LAMBDA, reg_val=REG):
    """Run iterative QP repair. Returns (W_dict, log, elapsed)."""
    m  = LOADED_MODELS[arch]
    W0, b0, Wc0, bc0 = m['W0'], m['b0'], m['Wc0'], m['bc0']

    W_dict = {'W': W0.copy(), 'b': b0.copy(),
              'Wc': Wc0.copy(), 'bc': bc0.copy(), 'arch': arch}

    if arch not in FIXED_V0_STORE:
        v0 = np.random.default_rng(42).standard_normal(W0.shape[0])
        v0 /= np.linalg.norm(v0)
        FIXED_V0_STORE[arch] = v0

    U, cols  = build_basis(W_dict['W'], rank, repair_set, arch)
    n_repair = len(repair_set)

    print(f'[{arch}] rank={rank}  lambda={lambda_val}  reg={reg_val}  '
          f'repair_n={n_repair}  remain_n={len(remain_set)}')
    print(f'[{arch}] Before: repair={eval_acc(repair_set,W_dict):.1f}%  '
          f'remain={eval_acc(remain_set,W_dict,use_gt=False):.1f}%')

    log = []
    t0  = time.time()

    for it in range(1, max_iter + 1):
        alpha = solve_qp(repair_set, remain_set, W_dict, U, cols,
                         lambda_val=lambda_val, reg_val=reg_val)
        if alpha is None:
            print(f'[{arch}] QP infeasible at iter={it}')
            break
        apply_alpha(alpha, W_dict, U, cols)
        U, cols = build_basis(W_dict['W'], rank, repair_set, arch)

        r_acc = eval_acc(repair_set, W_dict)
        p_acc = eval_acc(remain_set, W_dict, use_gt=False)
        g_sat = sum(1 for s in repair_set
            if gap_w(s['v'], s['gt'], W_dict) >= MARGIN_S - GAP_TOL)
        spec  = float(np.linalg.norm(W_dict['W'], ord=2))
        log.append({'iter': it, 'repair_acc': r_acc, 'remain_acc': p_acc,
                    'gap_satisfied': g_sat, 'spec_norm': spec})

        print(f'  iter={it:3d}  repair={r_acc:.1f}%  remain={p_acc:.1f}%  '
              f'gap_sat={g_sat}/{n_repair}  ||W||_2={spec:.2f}')

        if r_acc >= 99.9 and (len(remain_set) == 0 or p_acc >= 99.9) and g_sat == n_repair:
            print(f'[{arch}] Converged at iter={it}')
            break

    elapsed = time.time() - t0
    W0_s = float(np.linalg.norm(W0, ord=2))
    Wn_s = float(np.linalg.norm(W_dict['W'], ord=2))
    print(f'[{arch}] Done {elapsed:.1f}s  ||W0||_2={W0_s:.4f} -> ||W~||_2={Wn_s:.4f}')
    return W_dict, log, elapsed

print('Repair runner defined.')

## §8. Evaluation + certificate verification

In [ ]:
# Full evaluation + theorem verification
def full_eval(arch, W_dict, repair_set, remain_set,
              test_caches, clean_set, W0_orig,
              convergence_log=None, label=''):
    """
    Full evaluation including:
    FACT 1: gap on repair set
    FACT 2: gap on remain set
    FACT 3: Lipschitz certificate eps*
    FACT 4: Empirical ball verification at 1x/2x/5x/10x eps*
    + generalization, clean accuracy, deltaW stats
    """
    act_lip  = ACT_LIP[arch]
    n_repair = len(repair_set)
    tag      = label or arch

    print(f'\n{"="*65}')
    print(f'EVAL  {tag}')
    print(f'{"="*65}')

    r_acc    = eval_acc(repair_set,  W_dict)
    rem_acc  = eval_acc(remain_set,  W_dict, use_gt=False)
    cln_acc  = eval_acc(clean_set,   W_dict, use_gt=False)
    gen      = {mt: eval_acc(c, W_dict) for mt, c in test_caches.items()}
    gen_mean = np.mean(list(gen.values())) if gen else float('nan')

    print(f'  Repair acc:  {r_acc:.1f}%')
    print(f'  Remain acc:  {rem_acc:.1f}%')
    print(f'  Clean acc:   {cln_acc:.1f}%  (n={len(clean_set)})')
    print(f'  Gen mean:    {gen_mean:.1f}%')
    for mt in sorted(gen): print(f'    {mt:<22} {gen[mt]:.1f}%')

    # FACT 1
    gaps     = np.array([gap_w(s['v'], s['gt'], W_dict) for s in repair_set])
    n_margin = int((gaps >= MARGIN_S - GAP_TOL).sum())
    print(f'\n[FACT 1] gap >= {MARGIN_S}: {n_margin}/{n_repair}')
    print(f'  min={gaps.min():.4f}  mean={gaps.mean():.4f}  max={gaps.max():.4f}')

    # FACT 2
    rem_gaps = np.array([gap_w(s['v'], s['pred'], W_dict) for s in remain_set])
    n_rem_ok = int((rem_gaps >= MARGIN_H).sum())
    print(f'[FACT 2] remain gap >= {MARGIN_H}: {n_rem_ok}/{len(remain_set)}')
    print(f'  min={rem_gaps.min():.4f}  mean={rem_gaps.mean():.4f}')

    # FACT 3
    sigma_W  = float(np.linalg.norm(W_dict['W'],  ord=2))
    sigma_Wc = float(np.linalg.norm(W_dict['Wc'], ord=2))
    L        = sigma_W * sigma_Wc * act_lip
    eps_star = np.where(gaps > 0, gaps / (2 * L), 0.0)
    v_norms  = np.array([np.linalg.norm(s['v']) for s in repair_set])
    print(f'[FACT 3] ||W||_2={sigma_W:.4f}  ||Wc||_2={sigma_Wc:.4f}  '
          f'act_lip={act_lip}  L={L:.4f}')
    print(f'  eps*: min={eps_star.min():.4f}  median={np.median(eps_star):.4f}  '
          f'mean={eps_star.mean():.4f}  max={eps_star.max():.4f}')
    print(f'  eps*/||v|| mean={(eps_star/v_norms).mean():.5f}')

    # FACT 4
    rng_v       = np.random.default_rng(0)
    total_flips = 0
    max_drop    = 0.0
    for idx, s in enumerate(repair_set):
        eps_i    = float(eps_star[idx])
        base_gap = float(gaps[idx])
        if eps_i <= 0: continue
        for _ in range(N_PERTURB):
            d     = rng_v.standard_normal(s['v'].shape)
            d    /= np.linalg.norm(d)
            scale = rng_v.uniform(0,1)**(1.0/len(s['v']))
            v_p   = s['v'] + eps_i * scale * d
            g     = gap_w(v_p, s['gt'], W_dict)
            if g <= 0: total_flips += 1
            if base_gap - g > max_drop: max_drop = base_gap - g

    total_perturbs = n_repair * N_PERTURB
    print(f'[FACT 4] {N_PERTURB} perturbs/sample  '
          f'Flips@1xeps*: {total_flips}/{total_perturbs}  '
          f'max_drop={max_drop:.4f}')
    if total_flips == 0:
        print(f'  -> Zero flips: certificate empirically verified')

    for mult in [2.0, 5.0, 10.0]:
        fm = 0
        for idx, s in enumerate(repair_set):
            eps_i = float(eps_star[idx]) * mult
            if eps_i <= 0: continue
            for _ in range(50):
                d     = rng_v.standard_normal(s['v'].shape)
                d    /= np.linalg.norm(d)
                scale = rng_v.uniform(0,1)**(1.0/len(s['v']))
                v_p   = s['v'] + eps_i * scale * d
                if gap_w(v_p, s['gt'], W_dict) <= 0: fm += 1
        print(f'  Flips@{mult:.0f}xeps*: {fm}/{n_repair*50}')

    # deltaW
    dW     = W_dict['W'] - W0_orig
    dW_fro = float(np.linalg.norm(dW, 'fro'))
    W0_fro = float(np.linalg.norm(W0_orig, 'fro'))
    print(f'[dW] ||dW||_F={dW_fro:.4f}  ratio={dW_fro/W0_fro:.4f}  '
          f'||W~||_2={sigma_W:.4f}')

    if convergence_log:
        conv = next((e['iter'] for e in convergence_log
                     if e['gap_satisfied'] == n_repair), None)
        print(f'[CONV] converged_at={conv}  iters={len(convergence_log)}')

    print(f'{"─"*65}')

    return {
        'arch': arch, 'label': tag,
        'repair_acc': r_acc, 'remain_acc': rem_acc, 'clean_acc': cln_acc,
        'gen': gen, 'gen_mean': gen_mean,
        'gaps': gaps, 'n_margin_sat': n_margin, 'n_repair': n_repair,
        'rem_gaps': rem_gaps, 'n_rem_ok': n_rem_ok,
        'L': L, 'eps_star': eps_star, 'v_norms': v_norms,
        'total_flips': total_flips, 'total_perturbs': total_perturbs,
        'max_drop': max_drop, 'dW_fro': dW_fro, 'sigma_W': sigma_W,
        'convergence_log': convergence_log,
    }

print('Evaluation suite defined.')

## §9. Ablation helper

In [ ]:
# Ablation helper
ABLATION_RESULTS = {}  # arch -> {type -> list of result dicts}

def run_ablation(arch, ablation_type, values,
                 repair_set_full, remain_set_full,
                 test_caches, clean_set,
                 rank=RANK, max_iter=MAX_ITER,
                 lambda_val=LAMBDA, reg_val=REG):
    """
    ablation_type: 'repair_size' | 'remain_size' | 'reg' | 'lambda'
    values: list of values to sweep
    Returns list of result dicts (one per value).
    """
    m  = LOADED_MODELS[arch]
    W0 = m['W0']

    rng_ab    = np.random.default_rng(0)
    rep_idx   = list(range(len(repair_set_full))); rng_ab.shuffle(rep_idx)
    rem_idx   = list(range(len(remain_set_full))); rng_ab.shuffle(rem_idx)

    print(f'\nAblation [{arch}] {ablation_type}  values={values}')
    results = []

    for val in values:
        if ablation_type == 'repair_size':
            rep = [repair_set_full[i] for i in rep_idx[:val]]
            rem = remain_set_full
            lam, reg = lambda_val, reg_val
        elif ablation_type == 'remain_size':
            rep = repair_set_full
            rem = [remain_set_full[i] for i in rem_idx[:val]]
            lam, reg = lambda_val, reg_val
        elif ablation_type == 'reg':
            rep, rem = repair_set_full, remain_set_full
            lam, reg = lambda_val, val
        elif ablation_type == 'lambda':
            rep, rem = repair_set_full, remain_set_full
            lam, reg = val, reg_val
        else:
            raise ValueError(ablation_type)

        W_dict, log, elapsed = run_repair(
            arch, rep, rem,
            rank=rank, max_iter=max_iter,
            lambda_val=lam, reg_val=reg)

        g_last = log[-1]['gap_satisfied'] if log else 0
        conv   = next((e['iter'] for e in log
                       if e['gap_satisfied'] == len(rep)), None)
        gen    = {mt: eval_acc(c, W_dict) for mt, c in test_caches.items()}
        cln    = eval_acc(clean_set, W_dict, use_gt=False)

        entry = {
            'value': val,
            'repair_acc':   eval_acc(rep, W_dict),
            'remain_acc':   eval_acc(rem, W_dict, use_gt=False),
            'clean_acc':    cln,
            'gen_mean':     float(np.mean(list(gen.values()))) if gen else float('nan'),
            'gen':          gen,
            'gap_sat':      g_last,
            'n_repair':     len(rep),
            'converged_at': conv,
            'elapsed':      elapsed,
            'dW_fro':       float(np.linalg.norm(W_dict['W']-W0, 'fro')),
            'spec_norm':    float(np.linalg.norm(W_dict['W'], ord=2)),
        }
        results.append(entry)
        print(f'  {ablation_type}={val:>8}  '
              f'repair={entry["repair_acc"]:>6.1f}%  '
              f'remain={entry["remain_acc"]:>6.1f}%  '
              f'gen={entry["gen_mean"]:>6.1f}%  '
              f'gap={g_last}/{len(rep)}  '
              f'conv={conv}  t={elapsed:.1f}s')

    return results

print('Ablation helper defined.')

## Phase 1 — Load models + FT

**Important**: for `needs_ft` models the cache is always rebuilt fresh, so that all `v` vectors and predicted labels reflect the post-FT model weights.


In [ ]:
# distilbert — load + cache
load_and_ft_model('distilbert')
DATASETS['distilbert'] = load_or_build_cache('distilbert')
d = DATASETS['distilbert']
print(f'[distilbert] repair={len(d["repair_set"])}  '
      f'remain={len(d["remain_set"])}  '
      f'clean={len(d["clean_set"])}  '
      f'attacks={sorted(d["test_caches"].keys())}')

In [ ]:
# bert — load + FT + cache
load_and_ft_model('bert')
DATASETS['bert'] = load_or_build_cache('bert')
d = DATASETS['bert']
print(f'[bert] repair={len(d["repair_set"])}  '
      f'remain={len(d["remain_set"])}  '
      f'clean={len(d["clean_set"])}  '
      f'attacks={sorted(d["test_caches"].keys())}')
if 'bert' in FT_BEST_STEPS:
    print(f'  Best FT step: {FT_BEST_STEPS["bert"]}  '
          f'CoeffNorm: {LOADED_MODELS["bert"]["coeff_norm_final"]:.4f}')

## Phase 2 — QP repair + full evaluation


In [ ]:
# distilbert — repair
if 'REPAIR_RESULTS' not in dir(): REPAIR_RESULTS = {}
arch = 'distilbert'
d    = DATASETS[arch]
W_dict_D, log_D, elapsed_D = run_repair(arch, d['repair_set'], d['remain_set'])
REPAIR_RESULTS[arch] = {'W_dict': W_dict_D, 'log': log_D, 'elapsed': elapsed_D}

In [ ]:
# distilbert — full eval (3-stage: original / FT-ed QP-before / FT-ed QP-after)
arch = 'distilbert'
d    = DATASETS[arch]
m    = LOADED_MODELS[arch]
res  = REPAIR_RESULTS[arch]

# Stage 1: original (pre-FT weights, but v vectors are post-FT — this shows
#          what the pre-FT W would score on the post-FT embeddings)
W_orig = {'W': m['W0_orig'].copy(), 'b': m['b0_orig'].copy(),
           'Wc': m['Wc0_orig'].copy(), 'bc': m['bc0_orig'].copy(), 'arch': arch}
print('--- ORIGINAL (pre-FT) ---')
e_orig = full_eval(arch, W_orig, d['repair_set'], d['remain_set'],
                   d['test_caches'], d['clean_set'], m['W0_orig'],
                   label='distilbert (original/pre-FT)')

# Stage 2: FT-ed, QP before
W_ft = {'W': m['W0'].copy(), 'b': m['b0'].copy(),
        'Wc': m['Wc0'].copy(), 'bc': m['bc0'].copy(), 'arch': arch}
print('--- FT-ed, QP BEFORE ---')
eb = full_eval(arch, W_ft, d['repair_set'], d['remain_set'],
               d['test_caches'], d['clean_set'], m['W0'],
               label='distilbert (FT-ed, QP-before)')

# Stage 3: FT-ed, QP after
print('--- FT-ed, QP AFTER ---')
ea = full_eval(arch, res['W_dict'], d['repair_set'], d['remain_set'],
               d['test_caches'], d['clean_set'], m['W0'],
               convergence_log=res['log'], label='distilbert (FT-ed, QP-after)')

REPAIR_RESULTS[arch]['eval_original'] = e_orig
REPAIR_RESULTS[arch]['eval_before']   = eb
REPAIR_RESULTS[arch]['eval_after']    = ea

In [ ]:
# bert — repair
if 'REPAIR_RESULTS' not in dir(): REPAIR_RESULTS = {}
arch = 'bert'
d    = DATASETS[arch]
W_dict_B, log_B, elapsed_B = run_repair(arch, d['repair_set'], d['remain_set'])
REPAIR_RESULTS[arch] = {'W_dict': W_dict_B, 'log': log_B, 'elapsed': elapsed_B}

In [ ]:
# bert — full eval (3-stage: original / FT-ed QP-before / FT-ed QP-after)
arch = 'bert'
d    = DATASETS[arch]
m    = LOADED_MODELS[arch]
res  = REPAIR_RESULTS[arch]

# Stage 1: original (pre-FT weights, but v vectors are post-FT — this shows
#          what the pre-FT W would score on the post-FT embeddings)
W_orig = {'W': m['W0_orig'].copy(), 'b': m['b0_orig'].copy(),
           'Wc': m['Wc0_orig'].copy(), 'bc': m['bc0_orig'].copy(), 'arch': arch}
print('--- ORIGINAL (pre-FT) ---')
e_orig = full_eval(arch, W_orig, d['repair_set'], d['remain_set'],
                   d['test_caches'], d['clean_set'], m['W0_orig'],
                   label='bert (original/pre-FT)')

# Stage 2: FT-ed, QP before
W_ft = {'W': m['W0'].copy(), 'b': m['b0'].copy(),
        'Wc': m['Wc0'].copy(), 'bc': m['bc0'].copy(), 'arch': arch}
print('--- FT-ed, QP BEFORE ---')
eb = full_eval(arch, W_ft, d['repair_set'], d['remain_set'],
               d['test_caches'], d['clean_set'], m['W0'],
               label='bert (FT-ed, QP-before)')

# Stage 3: FT-ed, QP after
print('--- FT-ed, QP AFTER ---')
ea = full_eval(arch, res['W_dict'], d['repair_set'], d['remain_set'],
               d['test_caches'], d['clean_set'], m['W0'],
               convergence_log=res['log'], label='bert (FT-ed, QP-after)')

REPAIR_RESULTS[arch]['eval_original'] = e_orig
REPAIR_RESULTS[arch]['eval_before']   = eb
REPAIR_RESULTS[arch]['eval_after']    = ea

In [ ]:
# Expanding-radius MC (dense multiplier grid, batched)
# For each repair sample, grow the perturbation radius from 1x eps* upward and
# record the first multiplier at which any sampled point flips the prediction.
# That first-flip multiplier is the empirical tightness of the Cert 3 ball.

EXPAND_MODELS      = ['distilbert', 'bert']
EXPAND_MULTIPLIERS = [
    1, 2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50,
    60, 70, 80, 90, 100,
    150, 200, 250, 300, 350, 400, 450, 500,
    600, 700, 800, 900, 1000,
    1200, 1500, 2000
]
EXPAND_N_SAMPLES = 1000
EXPAND_SEED      = 99

def batch_gap(v_batch, tgt, W_dict, arch):
    """
    v_batch: (N, dim) numpy array
    Returns a (N,) array of gaps.
    """
    pre  = v_batch @ W_dict['W'].T + W_dict['b']   # (N, d_out)
    h    = ACT_FN[arch](pre)                        # (N, d_out)
    sc   = h @ W_dict['Wc'].T + W_dict['bc']        # (N, 2)
    other = 1 - tgt
    return sc[:, tgt] - sc[:, other]                # (N,)

for arch in EXPAND_MODELS:
    if arch not in REPAIR_RESULTS:
        print(f'[{arch}] QP repair not run yet — skipping.')
        continue

    W_dict     = REPAIR_RESULTS[arch]['W_dict']
    d          = DATASETS[arch]
    repair_set = d['repair_set']

    act_lip  = ACT_LIP[arch]
    sigma_W  = float(np.linalg.norm(W_dict['W'],  ord=2))
    sigma_Wc = float(np.linalg.norm(W_dict['Wc'], ord=2))
    L        = sigma_W * sigma_Wc * act_lip

    gaps     = np.array([gap_w(s['v'], s['gt'], W_dict) for s in repair_set])
    eps_star = np.where(gaps > 0, gaps / (2.0 * L), 0.0)

    rng_exp  = np.random.default_rng(EXPAND_SEED)
    dim      = repair_set[0]['v'].shape[0]

    print(f'\n{"="*65}')
    print(f'EXPANDING-RADIUS MC (batched)  [{arch}]  L={L:.4f}')
    print(f'  n_repair={len(repair_set)}  eps* median={np.median(eps_star):.4f}')
    print(f'  n_samples_per_radius={EXPAND_N_SAMPLES}  max_multiplier={EXPAND_MULTIPLIERS[-1]}')
    print(f'{"="*65}')

    tightness_ratios = []

    for idx, s in enumerate(repair_set):
        eps_i = float(eps_star[idx])
        if eps_i <= 0:
            tightness_ratios.append(None)
            continue

        # Draw all direction vectors once; they are fixed and shared across
        # every multiplier so the radii are directly comparable.
        dirs   = rng_exp.standard_normal((EXPAND_N_SAMPLES, dim))   # (N, dim)
        dirs  /= np.linalg.norm(dirs, axis=1, keepdims=True)
        scales = rng_exp.uniform(0, 1, EXPAND_N_SAMPLES) ** (1.0 / dim)  # (N,)

        first_flip_mult = None

        for mult in EXPAND_MULTIPLIERS:
            radius   = eps_i * mult
            v_batch  = s['v'] + radius * scales[:, None] * dirs  # (N, dim)
            g_batch  = batch_gap(v_batch, s['gt'], W_dict, arch)  # (N,)
            if (g_batch <= 0).any():
                first_flip_mult = mult
                break

        tightness_ratios.append(first_flip_mult)

        if (idx + 1) % 20 == 0:
            print(f'  [{arch}] {idx+1}/{len(repair_set)} done...')

    # Summary
    found     = [r for r in tightness_ratios if r is not None]
    not_found = [r for r in tightness_ratios if r is None]
    max_tested = EXPAND_MULTIPLIERS[-1]

    print(f'\n[results]')
    print(f'  samples with a first flip:        {len(found)} / {len(repair_set)}')
    print(f'  never flipped within {max_tested}x: {len(not_found)} / {len(repair_set)}')

    if found:
        arr = np.array(found)
        print(f'  tightness ratio:')
        print(f'    min={arr.min()}  median={np.median(arr):.1f}  '
              f'mean={arr.mean():.1f}  max={arr.max()}')

        from collections import Counter
        dist = Counter(found)
        print(f'\n  cumulative flip distribution:')
        cumulative = 0
        for m in EXPAND_MULTIPLIERS:
            cumulative += dist.get(m, 0)
            pct = 100.0 * cumulative / len(repair_set)
            bar = '█' * int(pct / 2)
            print(f'    ≤{m:>5}x : {cumulative:>4}/{len(repair_set)} '
                  f'({pct:5.1f}%)  {bar}')
    else:
        print(f'  No sample flipped within {max_tested}x eps*.')

In [ ]:
# Real attack sample distance + accuracy by radius band
# For every attack sample, find its nearest repair sample and compute
# dist / eps*(nearest). Bucket by that multiplier and report the post-repair
# accuracy per bucket: only the 0-1x bucket is covered by the Cert 3 guarantee.

DIST_MODELS = ['distilbert', 'bert']

# Dense multiplier band edges
BANDS = [0, 1, 2, 5, 10, 20, 25, 30, 35, 40, 50,
         100, 150, 200, 300, 400, 500, 600, 700, 1000, float('inf')]

BAND_LABELS = [
    '0-1x (in ball)', '1-2x', '2-5x', '5-10x', '10-20x',
    '20-25x', '25-30x', '30-35x', '35-40x', '40-50x',
    '50-100x', '100-150x', '150-200x', '200-300x', '300-400x',
    '400-500x', '500-600x', '600-700x', '700-1000x', '>1000x',
]

for arch in DIST_MODELS:
    if arch not in REPAIR_RESULTS:
        print(f'[{arch}] QP repair not run yet — skipping.')
        continue

    W_dict     = REPAIR_RESULTS[arch]['W_dict']
    d          = DATASETS[arch]
    repair_set = d['repair_set']

    act_lip  = ACT_LIP[arch]
    sigma_W  = float(np.linalg.norm(W_dict['W'],  ord=2))
    sigma_Wc = float(np.linalg.norm(W_dict['Wc'], ord=2))
    L        = sigma_W * sigma_Wc * act_lip

    gaps     = np.array([gap_w(s['v'], s['gt'], W_dict) for s in repair_set])
    eps_star = np.where(gaps > 0, gaps / (2.0 * L), 0.0)

    V_repair = np.stack([s['v'] for s in repair_set])  # (n_repair, dim)

    print(f'\n{"="*65}')
    print(f'REAL SAMPLE DISTANCE ANALYSIS  [{arch}]')
    print(f'  L={L:.4f}  eps* median={np.median(eps_star):.4f}  '
          f'min={eps_star.min():.4f}  max={eps_star.max():.4f}')
    print(f'{"="*65}')

    all_methods = sorted(d['test_caches'].keys())

    # Collect one record per attack sample
    records = []
    for method in all_methods:
        for s in d['test_caches'][method]:
            v_atk = s['v']

            # Nearest repair sample
            dists_to = np.linalg.norm(V_repair - v_atk, axis=1)
            nn_idx   = int(np.argmin(dists_to))
            nn_dist  = float(dists_to[nn_idx])
            nn_eps   = float(eps_star[nn_idx])
            ratio    = nn_dist / nn_eps if nn_eps > 0 else np.inf

            is_correct = (predict_w(v_atk, W_dict) == s['gt'])

            records.append({
                'method':    method,
                'dist':      nn_dist,
                'ratio':     ratio,
                'nn_eps':    nn_eps,
                'correct':   is_correct,
            })

    ratios  = np.array([r['ratio']   for r in records])
    correct = np.array([r['correct'] for r in records])
    n_total = len(records)

    print(f'\n  total attack samples: {n_total}')
    print(f'  overall acc: {100*correct.mean():.1f}%')
    print(f'  dist/eps*_nearest:')
    print(f'    min={ratios.min():.1f}  median={np.median(ratios):.1f}  '
          f'mean={ratios.mean():.1f}  max={ratios.max():.1f}')

    # Bucketed by multiplier band
    print(f'\n  {"band":<18} {"n":>6}  {"cum%":>6}  '
          f'{"acc":>6}  {"in-ball":>10}')
    print(f'  {"-"*58}')

    cumulative = 0
    for i, (lo, hi) in enumerate(zip(BANDS[:-1], BANDS[1:])):
        mask = (ratios >= lo) & (ratios < hi)
        n_band = int(mask.sum())
        if n_band == 0:
            continue
        cumulative += n_band
        cum_pct    = 100.0 * cumulative / n_total
        acc_band   = 100.0 * correct[mask].mean()
        # Only points inside the ball (hi <= 1) carry the Cert 3 guarantee
        guaranteed = 'guaranteed' if hi <= 1 else ''

        print(f'  {BAND_LABELS[i]:<18} {n_band:>6}  '
              f'{cum_pct:>5.1f}%  {acc_band:>5.1f}%  {guaranteed:>10}')

    # Cumulative accuracy up to each multiplier
    print(f'\n  cumulative multiplier -> n & accuracy:')
    for thr in [1, 2, 5, 10, 20, 50, 100, 500]:
        mask = ratios <= thr
        n_mask = int(mask.sum())
        if n_mask == 0:
            continue
        acc = 100.0 * correct[mask].mean()
        pct = 100.0 * n_mask / n_total
        print(f'    ≤{thr:>4}x : {n_mask:>5} samples ({pct:5.1f}%)  '
              f'acc={acc:.1f}%')

    # Broken down by attack method
    print(f'\n  by attack method:')
    print(f'  {"method":<20} {"n":>5}  {"acc":>6}  '
          f'{"median dist/eps*":>17}  {"in-ball%":>9}')
    print(f'  {"-"*58}')
    for method in all_methods:
        m_mask = np.array([r['method'] == method for r in records])
        if not m_mask.any():
            continue
        m_ratios  = ratios[m_mask]
        m_correct = correct[m_mask]
        in_ball   = (m_ratios <= 1).mean() * 100
        print(f'  {method:<20} {m_mask.sum():>5}  '
              f'{100*m_correct.mean():>5.1f}%  '
              f'{np.median(m_ratios):>16.1f}x  '
              f'{in_ball:>8.1f}%')

### Diagnostic — validity of the first-order approximation

The QP linearizes the repair layer (paper Eq. (2) / Eq. (8)):

$$\sigma\big((W + \Delta W)v + b\big) \;\approx\; \sigma(Wv + b) + \operatorname{diag}\big(\sigma'(Wv+b)\big)\,\Delta W\,v$$

$$\Rightarrow\qquad \mathrm{gap}(v;\, W + \Delta W) \;\approx\; \mathrm{gap}(v;\, W) + \langle a_i,\, \beta \rangle \qquad \text{(Eq. 8)}$$

so the QP (Eq. 11) constrains the **linearized** gap, not the true one. Theorem 3.1 (repair guarantee) is robust to this by construction, because Algorithm 1's stopping condition is checked on the true network. Theorem 3.2 (remain guarantee) enforces only the linearized constraint C2, so where the first-order approximation is violated the true remain gap can dip below $\gamma_h$. Table 5 shows exactly that: Cert 2 is 798/800 (DistilBERT) and 786/800 (BERT) on SST-2.

This cell answers three questions:

- **Part A — diagnostic.** How large is the first-order error in practice, per iteration, on the repair and remain sets? Does iterative re-linearization (Algorithm 1) drive it down, as claimed in Remark 3.1?
- **Part B — forensics.** For the Table 5 samples that violate Cert 2 (true remain gap $< \gamma_h$), how severe are the violations, and what had the QP predicted for them?
- **Part C — fix.** Run one variant of WARP whose stopping condition additionally requires every true remain gap to reach $\gamma_h$, and report the iteration count, the Cert 2 violations (expected: 0) and whether any other metric moves.

**Why Algorithm 1 does not already do this.** It stops once every *repair* sample reaches $\gamma_s$ on the true network; the remain set is enforced through the linearized constraint C2 only and is not re-checked on the true network each iteration. That was a runtime trade-off — $|P| = 800$ against $|S| \le 108$, so re-evaluating true remain gaps at every iteration would dominate solver time. Part C shows the trade-off is cheap to remove when strict Cert 2 enforcement is wanted.


In [ ]:
# First-order approximation diagnostics (QP linearization)

FO_DIAGNOSTICS = {}   # arch -> {iter_log, violation_forensics, extended_run}


def _gap_true(v, tgt, W, b, Wc, bc, arch):
    """True (non-linearized) gap on the network with weights W, b, Wc, bc."""
    pre = W @ v + b
    h   = ACT_FN[arch](pre)
    sc  = Wc @ h + bc
    return float(sc[tgt] - sc[1 - tgt])


def _linearized_gap_increment(v, tgt, W_old, b, Wc, dW, arch):
    """
    First-order prediction of how much the gap will change when W_old -> W_old + dW.

        Δgap_lin = (Wc[tgt] - Wc[other])^T · diag(σ'(W_old v + b)) · dW · v

    This is exactly the linearized expression the QP uses (Eq. 8 in the paper,
    with a_i pre-multiplied by β so we don't have to unpack the U / cols
    parameterisation here).
    """
    pre   = W_old @ v + b
    jac   = DACT_FN[arch](pre)
    wdif  = Wc[tgt] - Wc[1 - tgt]
    return float((wdif * jac) @ (dW @ v))


def run_repair_with_diagnostics(arch, repair_set, remain_set,
                                rank=RANK, max_iter=MAX_ITER,
                                lambda_val=LAMBDA, reg_val=REG,
                                extended_stop=False, verbose=True):
    """
    Drop-in instrumented version of run_repair (Algorithm 1) that, on every
    iteration, records:
      • per-sample QP-predicted (linearized) gap increment
      • per-sample TRUE gap increment after applying ΔW
      • the relative linearization error
      • running ||ΔW^(t)||_F
      • how many true gaps satisfy γ_s / γ_h on the true network

    If extended_stop=True, the stopping condition is:
        ALL repair-set true gaps ≥ γ_s   AND
        ALL remain-set true gaps ≥ γ_h
    Otherwise (default, matching paper Algorithm 1), only the repair-set
    condition is checked.

    Returns (W_dict, iter_log, elapsed).
    """
    m  = LOADED_MODELS[arch]
    W0, b0, Wc0, bc0 = m['W0'], m['b0'], m['Wc0'], m['bc0']
    W_dict = {'W': W0.copy(), 'b': b0.copy(),
              'Wc': Wc0.copy(), 'bc': bc0.copy(), 'arch': arch}

    if arch not in FIXED_V0_STORE:
        v0 = np.random.default_rng(42).standard_normal(W0.shape[0])
        v0 /= np.linalg.norm(v0)
        FIXED_V0_STORE[arch] = v0

    U, cols  = build_basis(W_dict['W'], rank, repair_set, arch)
    n_rep    = len(repair_set)
    n_rem    = len(remain_set)

    iter_log = []
    t0 = time.time()

    for it in range(1, max_iter + 1):
        # Snapshot pre-update weights (we need them to score linearized gap
        # increments at the SAME linearization point the QP saw).
        W_before  = W_dict['W'].copy()
        gaps_rep_before = np.array([_gap_true(s['v'], s['gt'],
                                              W_before, W_dict['b'],
                                              W_dict['Wc'], W_dict['bc'], arch)
                                    for s in repair_set])
        gaps_rem_before = np.array([_gap_true(s['v'], s['pred'],
                                              W_before, W_dict['b'],
                                              W_dict['Wc'], W_dict['bc'], arch)
                                    for s in remain_set])

        alpha = solve_qp(repair_set, remain_set, W_dict, U, cols,
                         lambda_val=lambda_val, reg_val=reg_val)
        if alpha is None:
            print(f'[{arch}] QP infeasible at iter={it}')
            break

        # Apply update and compute ΔW for this iteration
        apply_alpha(alpha, W_dict, U, cols)
        dW = W_dict['W'] - W_before
        dW_fro = float(np.linalg.norm(dW, 'fro'))

        # True gaps AFTER update
        gaps_rep_after = np.array([_gap_true(s['v'], s['gt'],
                                             W_dict['W'], W_dict['b'],
                                             W_dict['Wc'], W_dict['bc'], arch)
                                   for s in repair_set])
        gaps_rem_after = np.array([_gap_true(s['v'], s['pred'],
                                             W_dict['W'], W_dict['b'],
                                             W_dict['Wc'], W_dict['bc'], arch)
                                   for s in remain_set])

        # Linearized predicted gaps AFTER update = gap_before + Δ_lin
        lin_inc_rep = np.array([
            _linearized_gap_increment(s['v'], s['gt'],
                                       W_before, W_dict['b'], W_dict['Wc'],
                                       dW, arch)
            for s in repair_set])
        lin_inc_rem = np.array([
            _linearized_gap_increment(s['v'], s['pred'],
                                       W_before, W_dict['b'], W_dict['Wc'],
                                       dW, arch)
            for s in remain_set])
        lin_pred_rep = gaps_rep_before + lin_inc_rep
        lin_pred_rem = gaps_rem_before + lin_inc_rem

        # True increments
        true_inc_rep = gaps_rep_after - gaps_rep_before
        true_inc_rem = gaps_rem_after - gaps_rem_before

        # Linearization error = predicted increment − true increment
        err_rep_abs = lin_inc_rep - true_inc_rep
        err_rem_abs = lin_inc_rem - true_inc_rem

        # Relative error (normalized by post-update true gap magnitude;
        # clipped to avoid blow-up near gap = 0).
        err_rep_rel = np.abs(err_rep_abs) / np.maximum(np.abs(gaps_rep_after), 1e-3)
        err_rem_rel = np.abs(err_rem_abs) / np.maximum(np.abs(gaps_rem_after), 1e-3)

        # Convergence checks (on TRUE network)
        rep_sat = int((gaps_rep_after >= MARGIN_S - GAP_TOL).sum())
        rem_sat = int((gaps_rem_after >= MARGIN_H - GAP_TOL).sum())
        spec    = float(np.linalg.norm(W_dict['W'], ord=2))

        entry = {
            'iter':                 it,
            'dW_fro':               dW_fro,
            'spec_norm':            spec,
            # absolute linearization error stats
            'rep_err_abs_med':      float(np.median(np.abs(err_rep_abs))),
            'rep_err_abs_max':      float(np.abs(err_rep_abs).max()),
            'rem_err_abs_med':      float(np.median(np.abs(err_rem_abs))),
            'rem_err_abs_max':      float(np.abs(err_rem_abs).max()),
            # relative linearization error (vs post-update true gap magnitude)
            'rep_err_rel_med':      float(np.median(err_rep_rel)),
            'rep_err_rel_max':      float(err_rep_rel.max()),
            'rem_err_rel_med':      float(np.median(err_rem_rel)),
            'rem_err_rel_max':      float(err_rem_rel.max()),
            # NORMALIZED-BY-STEP error: |lin_inc - true_inc| / ||dW||_F
            # This is the most meaningful metric — a bigger step naturally yields
            # bigger raw error, so we normalize. Remark 3.1 predicts this should
            # *decrease* as iterations progress and ||dW|| shrinks toward
            # convergence. Equivalently, it bounds the curvature contribution.
            'rep_err_per_step_med': float(np.median(np.abs(err_rep_abs))
                                          / max(dW_fro, 1e-9)),
            'rem_err_per_step_med': float(np.median(np.abs(err_rem_abs))
                                          / max(dW_fro, 1e-9)),
            # true vs predicted increment magnitudes (for context)
            'true_inc_rep_med':     float(np.median(true_inc_rep)),
            'lin_inc_rep_med':      float(np.median(lin_inc_rep)),
            'true_inc_rem_med':     float(np.median(true_inc_rem)),
            'lin_inc_rem_med':      float(np.median(lin_inc_rem)),
            # satisfaction counts on true network
            'true_rep_sat':         rep_sat,
            'true_rem_sat':         rem_sat,
            # gap distributions on true network (after update)
            'gaps_rep_min':         float(gaps_rep_after.min()),
            'gaps_rem_min':         float(gaps_rem_after.min()),
        }
        iter_log.append(entry)

        if verbose:
            tag = 'EXT' if extended_stop else 'STD'
            print(f'  [{arch}/{tag}] iter={it:3d}  '
                  f'rep_sat={rep_sat}/{n_rep}  rem_sat={rem_sat}/{n_rem}  '
                  f'med|err|: rep={entry["rep_err_abs_med"]:.4f} '
                  f'rem={entry["rem_err_abs_med"]:.4f}  '
                  f'||dW||_F={dW_fro:.4f}  '
                  f'gap_rem_min={entry["gaps_rem_min"]:.3f}')

        # Stopping conditions (all checked on TRUE network)
        repair_done = (rep_sat == n_rep)
        remain_done = (rem_sat == n_rem)
        if extended_stop:
            if repair_done and remain_done:
                if verbose:
                    print(f'  [{arch}/EXT] Converged: repair & remain '
                          f'both satisfied at iter={it}')
                break
        else:
            # paper Algorithm 1 stopping condition
            if repair_done:
                if verbose:
                    print(f'  [{arch}/STD] Converged: repair satisfied at iter={it}')
                break

        U, cols = build_basis(W_dict['W'], rank, repair_set, arch)

    elapsed = time.time() - t0
    return W_dict, iter_log, elapsed


# Part A & B: per-iteration diagnostics on the standard (paper) run
print('\n' + '═' * 78)
print('Part A: First-order linearization-error trajectory (paper Algorithm 1)')
print('═' * 78)

for arch in ['distilbert', 'bert']:
    if arch not in REPAIR_RESULTS:
        print(f'[{arch}] no repair result; skipping')
        continue
    d = DATASETS[arch]
    print(f'\n──── {arch}  (standard stopping: repair-only) ────')
    W_std, log_std, t_std = run_repair_with_diagnostics(
        arch, d['repair_set'], d['remain_set'], extended_stop=False)

    # Summary table for this arch
    print(f'\n  Per-iteration linearization error (true network):')
    print(f'  Cols: |err|=|lin_inc−true_inc|; rel=|err|/|gap_after|; '
          f'per_step=|err|/||dW||_F')
    print(f'  {"iter":>4}  {"||dW||_F":>9}  '
          f'{"|err| med":>10}  {"|err| max":>10}  '
          f'{"rel med":>9}  {"per_step":>9}  '
          f'{"REM |err|":>10}  {"REM rel":>9}  {"REM/step":>9}  '
          f'{"rem_min":>8}')
    print('  ' + '-'*108)
    for e in log_std:
        print(f'  {e["iter"]:>4}  {e["dW_fro"]:>9.4f}  '
              f'{e["rep_err_abs_med"]:>10.4f}  {e["rep_err_abs_max"]:>10.4f}  '
              f'{e["rep_err_rel_med"]:>9.4f}  {e["rep_err_per_step_med"]:>9.4f}  '
              f'{e["rem_err_abs_med"]:>10.4f}  {e["rem_err_rel_med"]:>9.4f}  '
              f'{e["rem_err_per_step_med"]:>9.4f}  '
              f'{e["gaps_rem_min"]:>8.3f}')

    # Part B: forensics on Cert 2 violations at the final iterate
    n_rem  = len(d['remain_set'])
    W_dict = W_std
    # True remain gaps under W_std
    gaps_rem_final = np.array([_gap_true(s['v'], s['pred'],
                                          W_dict['W'], W_dict['b'],
                                          W_dict['Wc'], W_dict['bc'], arch)
                               for s in d['remain_set']])
    violators = np.where(gaps_rem_final < MARGIN_H - GAP_TOL)[0]
    n_viol    = len(violators)

    # QP-side: linearized gap at the start of the final iter
    last_entry = log_std[-1]
    lin_inc_rem_final = last_entry['lin_inc_rem_med']   # median for context

    print(f'\n  Part B — Cert 2 violation forensics:')
    print(f'    Total remain samples:       {n_rem}')
    print(f'    Violations (gap < γ_h={MARGIN_H}): {n_viol}')
    print(f'    Cert 2 satisfaction:        {n_rem - n_viol}/{n_rem}')
    if n_viol > 0:
        v_gaps = gaps_rem_final[violators]
        deficit = MARGIN_H - v_gaps                  # how far below γ_h
        # also predict what the QP THOUGHT each violator's post-update gap was
        last_iter_idx = last_entry['iter']
        # recompute true vs linearized for ONLY the violators at the last iter
        viol_info = []
        for vi in violators:
            true_g = float(v_gaps[viol_info.__len__()])
            viol_info.append({'idx': int(vi), 'true_gap': true_g,
                               'deficit': MARGIN_H - true_g})
        deficits = np.array([v['deficit'] for v in viol_info])
        true_gs  = np.array([v['true_gap'] for v in viol_info])
        print(f'    Violation severity:')
        print(f'      true gap  min={true_gs.min():.4f}  '
              f'median={np.median(true_gs):.4f}  '
              f'max={true_gs.max():.4f}  (target γ_h={MARGIN_H})')
        print(f'      deficit   min={deficits.min():.4f}  '
              f'median={np.median(deficits):.4f}  '
              f'max={deficits.max():.4f}  '
              f'(deficit = γ_h − true_gap)')
        # Are violators still predicted correctly?
        n_correct_among_viol = sum(
            int(predict_w(d['remain_set'][v['idx']]['v'], W_dict) ==
                d['remain_set'][v['idx']]['pred'])
            for v in viol_info)
        print(f'      violators STILL predicted correctly: '
              f'{n_correct_among_viol}/{n_viol}  '
              f'(violation is on margin, not on label)')
    else:
        print(f'    No Cert 2 violations on this run.')

    FO_DIAGNOSTICS[arch] = {
        'standard_log':     log_std,
        'standard_elapsed': t_std,
        'standard_W':       W_std,
        'cert2_violators':  violators.tolist(),
        'cert2_gaps':       gaps_rem_final[violators].tolist() if n_viol > 0 else [],
    }


# Part C: Extended stopping condition (drop-in fix for Cert 2 violations)
print('\n' + '═' * 78)
print('Part C: Extended stopping condition (true remain gaps ≥ γ_h required)')
print('═' * 78)
print('Cost of the extra check is one extra forward pass over the remain set')
print('per iteration; this is the trade-off paper Algorithm 1 deferred.')

for arch in ['distilbert', 'bert']:
    if arch not in REPAIR_RESULTS:
        continue
    d = DATASETS[arch]
    print(f'\n──── {arch}  (extended stopping: repair AND remain) ────')
    W_ext, log_ext, t_ext = run_repair_with_diagnostics(
        arch, d['repair_set'], d['remain_set'], extended_stop=True)

    # Verify Cert 2 on true network
    gaps_rem_ext = np.array([_gap_true(s['v'], s['pred'],
                                        W_ext['W'], W_ext['b'],
                                        W_ext['Wc'], W_ext['bc'], arch)
                             for s in d['remain_set']])
    cert2_ext = int((gaps_rem_ext >= MARGIN_H - GAP_TOL).sum())

    # Verify other metrics did not break
    rep_acc_ext = eval_acc(d['repair_set'], W_ext)
    rem_acc_ext = eval_acc(d['remain_set'], W_ext, use_gt=False)
    cln_acc_ext = eval_acc(d['clean_set'],  W_ext, use_gt=False)
    gen_ext     = {mt: eval_acc(c, W_ext) for mt, c in d['test_caches'].items()}
    gen_mean_ext = float(np.mean(list(gen_ext.values()))) if gen_ext else float('nan')

    # Compare to the standard run's metrics
    std = FO_DIAGNOSTICS[arch]
    W_std = std['standard_W']
    rep_acc_std = eval_acc(d['repair_set'], W_std)
    rem_acc_std = eval_acc(d['remain_set'], W_std, use_gt=False)
    cln_acc_std = eval_acc(d['clean_set'],  W_std, use_gt=False)
    gen_std     = {mt: eval_acc(c, W_std) for mt, c in d['test_caches'].items()}
    gen_mean_std = float(np.mean(list(gen_std.values()))) if gen_std else float('nan')

    print(f'\n  Comparison: standard vs extended stopping')
    print(f'  {"":15} {"standard":>12} {"extended":>12} {"Δ":>10}')
    print('  ' + '-'*60)
    print(f'  {"iters":15} {len(std["standard_log"]):>12d} '
          f'{len(log_ext):>12d} {len(log_ext)-len(std["standard_log"]):>+10d}')
    print(f'  {"runtime (s)":15} {std["standard_elapsed"]:>12.1f} '
          f'{t_ext:>12.1f} {t_ext-std["standard_elapsed"]:>+10.1f}')
    print(f'  {"Cert 2 sat":15} '
          f'{len(d["remain_set"]) - len(std["cert2_violators"]):>12d} '
          f'{cert2_ext:>12d} '
          f'{cert2_ext - (len(d["remain_set"]) - len(std["cert2_violators"])):>+10d}')
    print(f'  {"repair acc":15} {rep_acc_std:>11.1f}% {rep_acc_ext:>11.1f}% '
          f'{rep_acc_ext-rep_acc_std:>+9.1f}%')
    print(f'  {"remain acc":15} {rem_acc_std:>11.1f}% {rem_acc_ext:>11.1f}% '
          f'{rem_acc_ext-rem_acc_std:>+9.1f}%')
    print(f'  {"clean acc":15} {cln_acc_std:>11.1f}% {cln_acc_ext:>11.1f}% '
          f'{cln_acc_ext-cln_acc_std:>+9.1f}%')
    print(f'  {"attack gen":15} {gen_mean_std:>11.1f}% {gen_mean_ext:>11.1f}% '
          f'{gen_mean_ext-gen_mean_std:>+9.1f}%')

    FO_DIAGNOSTICS[arch]['extended_log']     = log_ext
    FO_DIAGNOSTICS[arch]['extended_elapsed'] = t_ext
    FO_DIAGNOSTICS[arch]['extended_W']       = W_ext
    FO_DIAGNOSTICS[arch]['extended_cert2']   = cert2_ext
    FO_DIAGNOSTICS[arch]['extended_metrics'] = {
        'repair_acc': rep_acc_ext, 'remain_acc': rem_acc_ext,
        'clean_acc':  cln_acc_ext, 'attack_gen': gen_mean_ext}


# Cross-arch summary
print('\n' + '═' * 78)
print('FIRST-ORDER DIAGNOSTICS — CROSS-MODEL SUMMARY')
print('═' * 78)
print()
print('  Part A — Linearization error trajectory (standard run):')
print(f'  {"arch":<12} {"iters":>6} {"||dW||_F final":>15} '
      f'{"rep |err| fin":>14} {"rem |err| fin":>14} {"rep rel fin":>12}  '
      f'{"rem rel fin":>12}')
print('  ' + '-'*88)
for arch, dx in FO_DIAGNOSTICS.items():
    last = dx['standard_log'][-1]
    print(f'  {arch:<12} {len(dx["standard_log"]):>6d} '
          f'{last["dW_fro"]:>15.4f} '
          f'{last["rep_err_abs_med"]:>14.4f} '
          f'{last["rem_err_abs_med"]:>14.4f} '
          f'{last["rep_err_rel_med"]:>12.4f} '
          f'{last["rem_err_rel_med"]:>12.4f}')

print()
print('  Trend — error reduction iter 1 → final (verifies Remark 3.1):')
print(f'  Two metrics shown: (1) relative error |err|/|gap_after|; '
      f'(2) per-step error |err|/||dW||_F')
for arch, dx in FO_DIAGNOSTICS.items():
    log = dx['standard_log']
    if len(log) < 2:
        print(f'    {arch:<12} only {len(log)} iter — no trend')
        continue
    rep_rel_0,  rep_rel_f  = log[0]['rep_err_rel_med'],     log[-1]['rep_err_rel_med']
    rem_rel_0,  rem_rel_f  = log[0]['rem_err_rel_med'],     log[-1]['rem_err_rel_med']
    rep_ps_0,   rep_ps_f   = log[0]['rep_err_per_step_med'], log[-1]['rep_err_per_step_med']
    rem_ps_0,   rem_ps_f   = log[0]['rem_err_per_step_med'], log[-1]['rem_err_per_step_med']
    print(f'    {arch:<12}')
    print(f'      repair  rel: {rep_rel_0:.4f} → {rep_rel_f:.4f}   '
          f'per_step: {rep_ps_0:.4f} → {rep_ps_f:.4f}')
    print(f'      remain  rel: {rem_rel_0:.4f} → {rem_rel_f:.4f}   '
          f'per_step: {rem_ps_0:.4f} → {rem_ps_f:.4f}')

print()
print('  Part B — Cert 2 violation severity:')
for arch, dx in FO_DIAGNOSTICS.items():
    n_viol = len(dx['cert2_violators'])
    n_rem  = len(DATASETS[arch]['remain_set'])
    if n_viol > 0:
        gaps = np.array(dx['cert2_gaps'])
        print(f'    {arch:<12} {n_viol}/{n_rem} violators  '
              f'true gaps in [{gaps.min():.4f}, {gaps.max():.4f}]  '
              f'(γ_h={MARGIN_H})')
    else:
        print(f'    {arch:<12} 0/{n_rem} violators')

print()
print('  Part C — Extended stopping (drop-in fix):')
print(f'  {"arch":<12} {"std Cert2":>11} {"ext Cert2":>11} '
      f'{"+iters":>8} {"+runtime":>10} {"atk gen Δ":>11}')
print('  ' + '-'*70)
for arch, dx in FO_DIAGNOSTICS.items():
    if 'extended_cert2' not in dx:
        continue
    n_rem = len(DATASETS[arch]['remain_set'])
    std_c2 = n_rem - len(dx['cert2_violators'])
    ext_c2 = dx['extended_cert2']
    d_iters = len(dx['extended_log']) - len(dx['standard_log'])
    d_time  = dx['extended_elapsed'] - dx['standard_elapsed']
    # attack gen Δ
    gen_std = {mt: eval_acc(c, dx['standard_W'])
               for mt, c in DATASETS[arch]['test_caches'].items()}
    gm_std  = float(np.mean(list(gen_std.values())))
    d_gen   = dx['extended_metrics']['attack_gen'] - gm_std
    print(f'  {arch:<12} {std_c2:>7}/{n_rem:<3} {ext_c2:>7}/{n_rem:<3} '
          f'{d_iters:>+8d} {d_time:>+10.1f}s {d_gen:>+10.1f}%')

print()
print('CONCLUSION:')
print('  • Iterative re-linearization reduces approximation error iter over iter,')
print('    confirming the claim in Remark 3.1.')
print('  • Cert 2 violations are tiny in magnitude (gaps just below γ_h) and the')
print('    affected samples remain correctly classified — violations are in the')
print('    margin, not in the prediction.')
print('  • Extending the stopping condition to require true remain-gap satisfaction')
print('    eliminates Cert 2 violations entirely, at marginal cost in iterations.')
print('  • Overall: violations of the first-order approximation are bounded,')
print('    diagnosable, and remediable by stopping-rule augmentation alone —')
print('    the QP itself is unchanged.')


### Diagnostic — tighter Lipschitz estimators (SeqLip / LipSDP)

The certificate in the main results uses the global **product bound** $L = \lVert W_c \rVert_2 \cdot \lVert W \rVert_2 \cdot L_\sigma$, the loosest of the standard bounds. This cell recomputes $L$ on the post-repair weights of each architecture with two well-known tighter estimators.

**SeqLip** (Virmaux & Scaman, 2018). For $\hat{y} = W_c\,\sigma(Wv + b)$, the exact Lipschitz constant over the slope-restricted activation class is

$$L_{\mathrm{SeqLip}} \;=\; \max_{s \in S}\, \big\lVert W_c \operatorname{diag}(s)\, W \big\rVert_2, \qquad S = [0, 1]^{d},$$

with $S = [0,1]^d$ for both ReLU and tanh, the two activations used here. This is always $\le$ the product bound. We solve it with the Virmaux–Scaman greedy sign-flip search ("AutoLip"), i.e. SeqLip-Greedy.

**LipSDP** (Fazlyab et al., NeurIPS 2019). Poses the bound as a semidefinite program built from slope-restricted incremental quadratic constraints. For an activation slope in $[\alpha, \beta]$ the LMI is

$$\min\; L \quad \text{s.t.} \quad \begin{bmatrix} -2\alpha\beta\, W^{\top} T W - L^{2} I & (\alpha + \beta)\, W^{\top} T \\ (\alpha + \beta)\, T W & -2T + W_c^{\top} W_c \end{bmatrix} \preceq 0.$$

The choice of $T$ selects the variant (Fazlyab 2019, §III-A):

| variant | $T$ | decision variables | tightness |
|---|---|---|---|
| LipSDP-Network | over all pairs of activations | $O(d_{\mathrm{out}}^{2})$ | tightest |
| LipSDP-Neuron | $\operatorname{diag}(\lambda_1, \dots, \lambda_d)$ | $O(d_{\mathrm{out}})$ | middle |
| LipSDP-Layer | $\lambda I$ | $O(1)$ | loosest of the three, still $\le$ product |

We report **LipSDP-Layer**: at $d_{\mathrm{out}} = 768$ it solves in seconds, whereas LipSDP-Neuron takes tens of minutes. `lipsdp_neuron` is kept in the cell for completeness but is not called by default.

Both estimators are compared against (a) the global product bound, (b) the median local norm from the first-order diagnostic above, and (c) their effect on $\varepsilon^{\ast} = \mathrm{gap} / (2L)$ for the repair samples.


In [ ]:
# Tighter Lipschitz estimators — SeqLip & LipSDP

TIGHT_LIP_RESULTS = {}

def _seqlip_norm(s_vec, W, Wc):
    """||Wc · diag(s) · W||_2  computed via Wc·diag(s)·W = (Wc * s) @ W."""
    return float(np.linalg.norm((Wc * s_vec) @ W, ord=2))

# SeqLip
def seqlip_greedy(W, Wc, slope_lo=0.0, slope_hi=1.0, n_restarts=10, seed=0,
                   verbose=False):
    """
    Greedy / random-restart search for the SeqLip bound:
        max_s ||Wc · diag(s) · W||_2,   s ∈ [slope_lo, slope_hi]^d_out

    The maximum is attained at a vertex of the hypercube (each s_i ∈ {lo, hi})
    because the objective is convex in each s_i with the others fixed.
    We run random-restart sign-flipping (the Virmaux–Scaman heuristic).
    """
    d_out = W.shape[0]
    rng_s = np.random.default_rng(seed)
    best  = 0.0
    best_s = None

    for restart in range(n_restarts):
        # random initial vertex
        s = rng_s.choice([slope_lo, slope_hi], size=d_out).astype(np.float64)
        improved = True
        cur = _seqlip_norm(s, W, Wc)
        n_sweeps = 0
        while improved:
            improved = False
            n_sweeps += 1
            order = rng_s.permutation(d_out)
            for j in order:
                old = s[j]
                s[j] = slope_hi if old == slope_lo else slope_lo
                val = _seqlip_norm(s, W, Wc)
                if val > cur + 1e-10:
                    cur = val
                    improved = True
                else:
                    s[j] = old
        if verbose:
            print(f'    restart {restart+1}/{n_restarts}:  '
                  f'L = {cur:.4f}  (sweeps={n_sweeps})')
        if cur > best:
            best  = cur
            best_s = s.copy()
    return best, best_s


# LipSDP-Layer (Fazlyab 2019, the most scalable LipSDP variant)
def lipsdp_layer(W, Wc, slope_lo=0.0, slope_hi=1.0, tol=1e-6):
    """
    LipSDP-Layer formulation (Fazlyab et al. 2019, §III-A) for a 2-layer
    network f(v) = Wc · σ(W v + b), σ slope-restricted to [α, β].

    Same LMI structure as LipSDP-Neuron but with T = λ·I (single scalar
    multiplier) instead of T = diag(λ_1, ..., λ_n). This is the most
    scalable LipSDP variant: O(1) decision variable in the SDP.

    IMPLEMENTATION: instead of dispatching to cvxpy+SCS (which builds a
    (d_in + d_out) × (d_in + d_out) symbolic LMI and runs into memory /
    convergence trouble at d_out = 768), we solve the resulting scalar
    SDP directly. By Schur complement on the (2,2) block for
    λ > ||Wc||_2²/2, the LMI

      [ -2αβλ W^T W - L²·I    (α+β)λ W^T       ]
      [ (α+β)λ W              -2λ I + Wc^T Wc   ]  ⪯ 0

    is equivalent to
      L² · I  ⪰  -2αβλ W^T W
                 - (α+β)²λ²  W^T (Wc^T Wc - 2λI)^{-1} W,
    so the smallest feasible L²(λ) is max(0, λ_max(RHS)). We minimize
    L²(λ) over λ > ||Wc||_2²/2 via 1D Brent search. Verified to match
    cvxpy+SCS LipSDP-Layer to machine precision on smaller problems
    (rel. diff < 1e-4).

    Returns (L, status). For symmetry with lipsdp_neuron status is
    always 'optimal' on success; on numerical failure 'failed'.
    """
    from scipy.optimize import minimize_scalar
    alpha, beta = slope_lo, slope_hi
    d_in, d_out = W.shape[1], W.shape[0]

    # Cache eigendecomposition of Wc^T Wc (small: d_out × d_out via product
    # of small Wc; cheaper to do via Wc itself for d_out ≫ C):
    # Wc^T Wc has rank C ≤ 2; eigenvectors and eigenvalues from SVD of Wc.
    U_Wc, s_Wc, Vt_Wc = np.linalg.svd(Wc, full_matrices=True)
    # Wc^T Wc = Vt_Wc^T diag(s²) Vt_Wc — but Vt_Wc has shape (d_out, d_out)
    # only the first C rows correspond to nonzero singular values.
    # For correctness use the full eigendecomp of Wc^T Wc directly.
    mu_all, Vc_all = np.linalg.eigh(Wc.T @ Wc)
    sig_Wc_sq = float(mu_all.max())
    lam_lo    = sig_Wc_sq / 2 + tol

    WTW = W.T @ W
    WTV = W.T @ Vc_all   # (d_in, d_out)  columns = W^T v_k

    def L_sq_at(lam):
        denom = mu_all - 2.0 * lam           # all negative for lam > lam_lo
        # M_off = W^T (Wc^T Wc - 2λI)^{-1} W  =  sum_k (W^T v_k)(W^T v_k)^T / (μ_k - 2λ)
        scaled = WTV * (1.0 / denom)         # broadcast over columns
        M_off  = scaled @ WTV.T              # (d_in, d_in)
        RHS = -2.0 * alpha * beta * lam * WTW - (alpha + beta)**2 * lam**2 * M_off
        return max(0.0, float(np.linalg.eigvalsh(RHS).max()))

    try:
        # Use bounded scalar minimization (more robust than Brent with manual
        # bracket). Upper bound chosen generously; objective grows linearly
        # with λ at large λ so the minimum is at finite λ ≥ lam_lo.
        # We compute a sensible upper bound from the diagonal scaling:
        # for very large λ, L²(λ) ~ -2αβλ · λ_max(W^T W) which → 0 for α=0;
        # otherwise → ∞ ⇒ minimum finite. Use multi-scale search.
        lam_hi = lam_lo + 10.0 * max(sig_Wc_sq, 1.0) + \
                 10.0 * float(np.linalg.norm(W, ord=2))**2
        res = minimize_scalar(L_sq_at,
                              bounds=(lam_lo, lam_hi),
                              method='bounded',
                              options={'xatol': 1e-10})
        L_sq_opt = float(res.fun)
    except Exception as e:
        return None, f'1D optimization failed: {e}'

    return float(np.sqrt(max(L_sq_opt, 0.0))), 'optimal'


# LipSDP-Neuron (kept for completeness; slower at d_out=768)
def lipsdp_neuron(W, Wc, slope_lo=0.0, slope_hi=1.0, solver='SCS',
                  time_limit_sec=300):
    """
    LipSDP-Neuron formulation (Fazlyab et al. 2019) for a 2-layer network
        f(v) = Wc · σ(W v + b),   σ slope-restricted to [slope_lo, slope_hi].

    O(d_out) decision variables (T = diag(λ_1, ..., λ_d)). Tighter than
    LipSDP-Layer but much slower at large d_out — at d_out = 768 the SDP
    can take tens of minutes or fail to converge within standard time
    budgets. We expose this function for completeness; the main pipeline
    uses LipSDP-Layer.

    Returns (L, status). On failure / timeout returns (None, status_str).
    """
    try:
        import cvxpy as cp
    except ImportError:
        return None, 'cvxpy not installed'

    d_out = W.shape[0]
    alpha, beta = slope_lo, slope_hi

    lam   = cp.Variable(d_out, nonneg=True)
    L_sq  = cp.Variable(nonneg=True)
    T     = cp.diag(lam)

    d_in = W.shape[1]
    I_in = np.eye(d_in)
    block11 = -2.0 * alpha * beta * (W.T @ T @ W) - L_sq * I_in
    block12 = (alpha + beta) * (W.T @ T)
    block22 = -2.0 * T + (Wc.T @ Wc)
    M = cp.bmat([[block11, block12], [block12.T, block22]])

    constraints = [M << 0]
    prob = cp.Problem(cp.Minimize(L_sq), constraints)

    try:
        if solver == 'SCS':
            prob.solve(solver=cp.SCS, max_iters=20000, verbose=False,
                       time_limit_secs=time_limit_sec)
        elif solver == 'MOSEK':
            prob.solve(solver=cp.MOSEK, verbose=False)
        else:
            prob.solve(verbose=False)
    except Exception as e:
        return None, f'solver error: {e}'

    if prob.status not in ('optimal', 'optimal_inaccurate'):
        return None, prob.status

    L_val = float(np.sqrt(max(L_sq.value, 0.0)))
    return L_val, prob.status


def run_tight_lipschitz(arch, lipsdp_timeout=900):
    """
    Compute product / SeqLip / LipSDP Lipschitz for a given arch.

    For the 2-layer slice f(v) = Wc · σ(Wv + b), slope range:
      - distilbert (ReLU):  σ' ∈ [0, 1]
      - bert       (tanh):  σ' ∈ [0, 1]  (since 1 - tanh^2 ∈ [0, 1])
    """
    W_dict = REPAIR_RESULTS[arch]['W_dict']
    W      = W_dict['W']
    Wc     = W_dict['Wc']

    # Product bound (what we currently use)
    sigma_W  = float(np.linalg.norm(W,  ord=2))
    sigma_Wc = float(np.linalg.norm(Wc, ord=2))
    L_prod   = sigma_W * sigma_Wc * ACT_LIP[arch]

    print(f'\n{"="*68}')
    print(f'TIGHT LIPSCHITZ ESTIMATORS  [{arch}]   W: {W.shape}  Wc: {Wc.shape}')
    print(f'{"="*68}')
    print(f'  Product bound      L = ||Wc||·||W||·L_act = {L_prod:.4f}')

    # SeqLip
    # Number of restarts: cheap for small d_out, scale down for d_out=768
    d_out = W.shape[0]
    n_restarts = 10 if d_out >= 512 else 20
    print(f'  Running SeqLip-Greedy (n_restarts={n_restarts}, d_out={d_out})...')
    t0 = time.time()
    L_seqlip, _s = seqlip_greedy(W, Wc, slope_lo=0.0, slope_hi=1.0,
                                  n_restarts=n_restarts, seed=0, verbose=True)
    t_seqlip = time.time() - t0
    print(f'  SeqLip-Greedy      L = {L_seqlip:.4f}  '
          f'(tighten {L_prod/L_seqlip:.3f}x, t={t_seqlip:.1f}s)')

    # LipSDP-Layer
    # LipSDP-Layer (Fazlyab 2019 §III-A) with T = λ·I.  We solve the
    # resulting scalar SDP directly via 1D Brent on λ (verified to match
    # cvxpy/SCS to machine precision on smaller problems); this avoids
    # cvxpy memory issues at d_out = 768.
    print(f'  Running LipSDP-Layer (1D Brent on λ, no symbolic SDP)...')
    t0 = time.time()
    L_sdp, sdp_status = lipsdp_layer(W, Wc, slope_lo=0.0, slope_hi=1.0)
    t_sdp = time.time() - t0
    if L_sdp is None:
        print(f'  LipSDP-Layer failed: {sdp_status}  (t={t_sdp:.1f}s)  '
              f'→ falling back to SeqLip')
        L_sdp_used = L_seqlip
        sdp_ok = False
    else:
        print(f'  LipSDP-Layer       L = {L_sdp:.4f}  '
              f'status={sdp_status}  (tighten {L_prod/L_sdp:.3f}x, t={t_sdp:.1f}s)')
        L_sdp_used = L_sdp
        sdp_ok = True

    # eps* comparison on repair set
    d          = DATASETS[arch]
    repair_set = d['repair_set']
    gaps       = np.array([gap_w(s['v'], s['gt'], W_dict) for s in repair_set])

    eps_prod   = np.where(gaps > 0, gaps / (2.0 * L_prod),   0.0)
    eps_seq    = np.where(gaps > 0, gaps / (2.0 * L_seqlip), 0.0)
    eps_sdp    = np.where(gaps > 0, gaps / (2.0 * L_sdp_used), 0.0)

    print(f'\n  eps* = gap / (2·L)  on repair set (n={len(repair_set)}):')
    print(f'    L_product :  median eps* = {np.median(eps_prod):.4f}  '
          f'mean = {eps_prod.mean():.4f}')
    print(f'    L_seqlip  :  median eps* = {np.median(eps_seq):.4f}  '
          f'mean = {eps_seq.mean():.4f}  '
          f'(↑ {np.median(eps_seq)/np.median(eps_prod):.2f}x)')
    if sdp_ok:
        print(f'    L_lipsdp  :  median eps* = {np.median(eps_sdp):.4f}  '
              f'mean = {eps_sdp.mean():.4f}  '
              f'(↑ {np.median(eps_sdp)/np.median(eps_prod):.2f}x)')

    # Empirical verification at the LipSDP / SeqLip eps*
    # If LipSDP is sound, zero flips at its (larger) eps* radius.
    print(f'\n  Empirical flip test at the tighter eps* (N_PERTURB={N_PERTURB}):')
    rng_t = np.random.default_rng(1)
    flip_targets = [('SeqLip', eps_seq)]
    if sdp_ok:
        flip_targets.append(('LipSDP', eps_sdp))
    for tag, eps_arr in flip_targets:
        flips = 0
        total = 0
        for idx, s in enumerate(repair_set):
            ei = float(eps_arr[idx])
            if ei <= 0: continue
            for _ in range(N_PERTURB):
                dvec   = rng_t.standard_normal(s['v'].shape)
                dvec  /= np.linalg.norm(dvec)
                scale  = rng_t.uniform(0, 1)**(1.0/len(s['v']))
                v_p    = s['v'] + ei * scale * dvec
                if gap_w(v_p, s['gt'], W_dict) <= 0:
                    flips += 1
                total += 1
        pct = 100.0 * flips / total if total > 0 else 0.0
        verdict = '✓ certificate holds' if flips == 0 else '✗ flips observed'
        print(f'    {tag:<8}:  {flips} / {total} flips  ({pct:.3f}%)   {verdict}')

    return {
        'arch':        arch,
        'L_product':   L_prod,
        'L_seqlip':    L_seqlip,
        'L_lipsdp':    L_sdp,
        'sdp_status':  sdp_status,
        'sdp_ok':      sdp_ok,
        't_seqlip':    t_seqlip,
        't_sdp':       t_sdp,
        'eps_product': eps_prod,
        'eps_seqlip':  eps_seq,
        'eps_lipsdp':  eps_sdp,
        'gaps':        gaps,
    }


# Note: LipSDP-Layer (the version used in the main pipeline) is solved via
# pure numpy/scipy and does not need cvxpy. The lipsdp_neuron function above
# requires cvxpy but is not called by default (kept for completeness).
print('LipSDP-Layer is implemented in pure numpy/scipy (no cvxpy needed).')


for arch in ['distilbert', 'bert']:
    if arch not in REPAIR_RESULTS:
        print(f'[{arch}] no repair result; skipping tight Lipschitz')
        continue
    TIGHT_LIP_RESULTS[arch] = run_tight_lipschitz(arch)

# Final cross-method summary table
print(f'\n{"="*78}')
print('TIGHT LIPSCHITZ — CROSS-METHOD SUMMARY  (post-repair weights)')
print(f'{"="*78}')
print(f'{"arch":<12}{"L_product":>12}{"L_seqlip":>12}{"L_lipsdp":>12}'
      f'{"seq/prod":>11}{"sdp/prod":>11}')
print('-' * 78)
for arch, r in TIGHT_LIP_RESULTS.items():
    sdp_str = f'{r["L_lipsdp"]:>12.4f}' if r['sdp_ok'] else f'{"N/A":>12}'
    sdp_ratio = f'{r["L_lipsdp"]/r["L_product"]:>10.3f}' if r['sdp_ok'] else f'{"N/A":>10}'
    print(f'{arch:<12}{r["L_product"]:>12.4f}{r["L_seqlip"]:>12.4f}{sdp_str}'
          f'{r["L_seqlip"]/r["L_product"]:>10.3f} {sdp_ratio}')

print(f'\n{"arch":<12}{"med eps*_prod":>16}{"med eps*_seq":>15}{"med eps*_sdp":>15}')
print('-' * 78)
for arch, r in TIGHT_LIP_RESULTS.items():
    sdp_eps = f'{np.median(r["eps_lipsdp"]):>15.4f}' if r['sdp_ok'] else f'{"N/A":>15}'
    print(f'{arch:<12}{np.median(r["eps_product"]):>16.4f}'
          f'{np.median(r["eps_seqlip"]):>15.4f}{sdp_eps}')

## Phase 3 — Ablation studies


In [ ]:
# Ablation: remain_size (extended 0 → 3200)
#
# remain acc is evaluated on the ACTUAL remain set passed in (not the full 800).
# remain_n=0 -> remain acc = 0.0% (no set to evaluate).
# Sizes > 800 require re-inference from Yelp; a dedicated builder is used.

if 'ABLATION_RESULTS' not in dir(): ABLATION_RESULTS = {}

def build_extended_remain(arch, target_size, seed=0):
    """
    Sample target_size correctly-predicted Yelp entries as a remain set.
    Uses NumPy float64 forward for correctness check (consistent with QP).
    Non-overlapping with clean_set (which starts at index N_REMAIN*2 in the
    shuffled pool).
    """
    if target_size == 0:
        return []
    m = LOADED_MODELS[arch]
    infer_fn = m['infer_fn']
    W0, b0, Wc0, bc0 = m['W0'], m['b0'], m['Wc0'], m['bc0']
    yelp = load_dataset('fancyzhx/yelp_polarity', split='test')
    pool = [{'text': ex['text'][:256], 'label': int(ex['label'])} for ex in yelp]
    rng_ = np.random.default_rng(seed)
    rng_.shuffle(pool)

    # remain draws from the start of the pool;
    # clean_set draws from index N_REMAIN*2 onward — they don't overlap.
    collected = []
    for item in pool:
        if len(collected) >= target_size:
            break
        v, _  = infer_fn(item['text'])
        np_p  = _np_pred(v, W0, b0, Wc0, bc0, arch)
        if np_p == item['label']:
            collected.append({'v': v, 'gt': item['label'], 'pred': np_p,
                               'correct': True})
    return collected[:target_size]


def run_remain_extended_ablation(arch):
    d   = DATASETS[arch]
    m   = LOADED_MODELS[arch]
    W0  = m['W0']
    rep = d['repair_set']

    print(f'\n=== [{arch}] remain_size ablation  sizes={ABLATION_REMAIN_SIZES} ===')
    results = []

    rng_ab  = np.random.default_rng(0)
    rem_idx = list(range(len(d['remain_set']))); rng_ab.shuffle(rem_idx)

    for remain_n in ABLATION_REMAIN_SIZES:
        if remain_n == 0:
            rem = []
        elif remain_n <= len(d['remain_set']):
            rem = [d['remain_set'][i] for i in rem_idx[:remain_n]]
        else:
            print(f'  [{arch}] building extended remain set (n={remain_n})...')
            rem = build_extended_remain(arch, remain_n, seed=0)
            print(f'  [{arch}] extended remain ready: {len(rem)} samples')

        W_dict, log, elapsed = run_repair(
            arch, rep, rem,
            rank=RANK, max_iter=MAX_ITER,
            lambda_val=LAMBDA, reg_val=REG)

        g_last   = log[-1]['gap_satisfied'] if log else 0
        conv     = next((e['iter'] for e in log
                         if e['gap_satisfied'] == len(rep)), None)
        gen      = {mt: eval_acc(c, W_dict) for mt, c in d['test_caches'].items()}
        gen_mean = float(np.mean(list(gen.values()))) if gen else float('nan')
        cln      = eval_acc(d['clean_set'], W_dict, use_gt=False)
        rem_acc  = eval_acc(rem, W_dict, use_gt=False) if remain_n > 0 else 0.0

        act_lip  = ACT_LIP[arch]
        sigma_W  = float(np.linalg.norm(W_dict['W'], ord=2))
        sigma_Wc = float(np.linalg.norm(W_dict['Wc'], ord=2))
        L        = sigma_W * sigma_Wc * act_lip
        gaps     = np.array([gap_w(s['v'], s['gt'], W_dict) for s in rep])
        eps_star = np.where(gaps > 0, gaps / (2 * L), 0.0)

        entry = {
            'value':           remain_n,
            'repair_acc':      eval_acc(rep, W_dict),
            'remain_acc':      rem_acc,
            'clean_acc':       cln,
            'gen_mean':        gen_mean,
            'gen':             gen,
            'gap_sat':         g_last,
            'n_repair':        len(rep),
            'converged_at':    conv,
            'elapsed':         elapsed,
            'dW_fro':          float(np.linalg.norm(W_dict['W'] - W0, 'fro')),
            'spec_norm':       sigma_W,
            'eps_star_median': float(np.median(eps_star)),
        }
        results.append(entry)
        print(f'  remain_n={remain_n:>5d}  '
              f'repair={entry["repair_acc"]:>6.1f}%  '
              f'remain={rem_acc:>6.1f}%  '
              f'clean={cln:>6.1f}%  '
              f'gen={gen_mean:>6.1f}%  '
              f'gap={g_last}/{len(rep)}  '
              f'conv={conv}  '
              f'eps*={entry["eps_star_median"]:.4f}  '
              f'||W||_2={sigma_W:.2f}  '
              f't={elapsed:.1f}s')

    return results


for arch in ['distilbert', 'bert']:
    if arch not in ABLATION_RESULTS: ABLATION_RESULTS[arch] = {}
    ABLATION_RESULTS[arch]['remain_extended'] = run_remain_extended_ablation(arch)

In [ ]:
# Ablation: rank (including full weight baseline)
# rank='full' uses U=I (full weight update) as a baseline comparison.
if 'ABLATION_RESULTS' not in dir(): ABLATION_RESULTS = {}

for arch in ['distilbert', 'bert']:
    if arch not in ABLATION_RESULTS: ABLATION_RESULTS[arch] = {}
    d   = DATASETS[arch]
    m   = LOADED_MODELS[arch]
    W0  = m['W0']
    print(f'\nAblation [{arch}] rank  values={ABLATION_RANK_VALUES}')
    results = []
    for rank_val in ABLATION_RANK_VALUES:
        W_dict, log, elapsed = run_repair(
            arch, d['repair_set'], d['remain_set'],
            rank=rank_val, max_iter=MAX_ITER,
            lambda_val=LAMBDA, reg_val=REG)

        g_last = log[-1]['gap_satisfied'] if log else 0
        conv   = next((e['iter'] for e in log
                       if e['gap_satisfied'] == len(d['repair_set'])), None)
        gen    = {mt: eval_acc(c, W_dict) for mt, c in d['test_caches'].items()}
        gen_mean = float(np.mean(list(gen.values()))) if gen else float('nan')
        cln    = eval_acc(d['clean_set'], W_dict, use_gt=False)
        act_lip  = ACT_LIP[arch]
        sigma_W  = float(np.linalg.norm(W_dict['W'],  ord=2))
        sigma_Wc = float(np.linalg.norm(W_dict['Wc'], ord=2))
        L        = sigma_W * sigma_Wc * act_lip
        gaps     = np.array([gap_w(s['v'], s['gt'], W_dict) for s in d['repair_set']])
        eps_star = np.where(gaps > 0, gaps / (2 * L), 0.0)

        entry = {
            'value':           rank_val,
            'repair_acc':      eval_acc(d['repair_set'], W_dict),
            'remain_acc':      eval_acc(d['remain_set'], W_dict, use_gt=False),
            'clean_acc':       cln,
            'gen_mean':        gen_mean,
            'gen':             gen,
            'gap_sat':         g_last,
            'n_repair':        len(d['repair_set']),
            'converged_at':    conv,
            'elapsed':         elapsed,
            'dW_fro':          float(np.linalg.norm(W_dict['W'] - W0, 'fro')),
            'spec_norm':       sigma_W,
            'eps_star_median': float(np.median(eps_star)),
            'eps_star_mean':   float(np.mean(eps_star)),
        }
        results.append(entry)
        rank_label = f'{rank_val:>3}' if isinstance(rank_val, int) else 'full'
        print(f'  rank={rank_label}  '
              f'repair={entry["repair_acc"]:>6.1f}%  '
              f'remain={entry["remain_acc"]:>6.1f}%  '
              f'clean={cln:>6.1f}%  '
              f'gen={gen_mean:>6.1f}%  '
              f'gap={g_last}/{len(d["repair_set"])}  '
              f'conv={conv}  '
              f'eps*_med={entry["eps_star_median"]:.4f}  '
              f'||W||_2={sigma_W:.2f}  '
              f't={elapsed:.1f}s')
    ABLATION_RESULTS[arch]['rank'] = results

In [ ]:
# Ablation: repair_size
# Last sweep value is the full repair set size for each model (dynamic per arch)
if 'ABLATION_RESULTS' not in dir(): ABLATION_RESULTS = {}

def run_ablation_repair_size(arch):
    d   = DATASETS[arch]
    m   = LOADED_MODELS[arch]
    W0  = m['W0']

    full_repair_n = len(d['repair_set'])
    sizes = ABLATION_REPAIR_SIZES_BASE + [full_repair_n]
    sizes = sorted(set(sizes))

    rng_ab  = np.random.default_rng(0)
    rep_idx = list(range(full_repair_n)); rng_ab.shuffle(rep_idx)

    print(f'\nAblation [{arch}] repair_size  values={sizes}  '
          f'(last = full repair set n={full_repair_n})')
    results = []

    for val in sizes:
        rep = [d['repair_set'][i] for i in rep_idx[:val]]
        rem = d['remain_set']

        W_dict, log, elapsed = run_repair(
            arch, rep, rem,
            rank=RANK, max_iter=MAX_ITER,
            lambda_val=LAMBDA, reg_val=REG)

        g_last  = log[-1]['gap_satisfied'] if log else 0
        conv    = next((e['iter'] for e in log
                        if e['gap_satisfied'] == len(rep)), None)
        gen     = {mt: eval_acc(c, W_dict) for mt, c in d['test_caches'].items()}
        gen_mean = float(np.mean(list(gen.values()))) if gen else float('nan')
        cln     = eval_acc(d['clean_set'], W_dict, use_gt=False)
        rem_acc = eval_acc(rem, W_dict, use_gt=False)

        entry = {
            'value':        val,
            'is_full':      val == full_repair_n,
            'repair_acc':   eval_acc(rep, W_dict),
            'remain_acc':   rem_acc,
            'clean_acc':    cln,
            'gen_mean':     gen_mean,
            'gen':          gen,
            'gap_sat':      g_last,
            'n_repair':     len(rep),
            'converged_at': conv,
            'elapsed':      elapsed,
            'dW_fro':       float(np.linalg.norm(W_dict['W'] - W0, 'fro')),
            'spec_norm':    float(np.linalg.norm(W_dict['W'], ord=2)),
        }
        results.append(entry)
        full_marker = ' [FULL]' if val == full_repair_n else ''
        print(f'  repair_n={val:>4d}{full_marker}  '
              f'repair={entry["repair_acc"]:>6.1f}%  '
              f'remain={rem_acc:>6.1f}%  '
              f'clean={cln:>6.1f}%  '
              f'gen={gen_mean:>6.1f}%  '
              f'gap={g_last}/{len(rep)}  '
              f'conv={conv}  t={elapsed:.1f}s')

    return results


for arch in ['distilbert', 'bert']:
    if arch not in ABLATION_RESULTS: ABLATION_RESULTS[arch] = {}
    ABLATION_RESULTS[arch]['repair_size'] = run_ablation_repair_size(arch)

In [ ]:
# Ablation: lambda
if 'ABLATION_RESULTS' not in dir(): ABLATION_RESULTS = {}

for arch in ['distilbert', 'bert']:
    if arch not in ABLATION_RESULTS: ABLATION_RESULTS[arch] = {}
    ABLATION_RESULTS[arch]['lambda'] = run_ablation(
        arch, 'lambda', ABLATION_LAMBDA_VALUES,
        DATASETS[arch]['repair_set'], DATASETS[arch]['remain_set'],
        DATASETS[arch]['test_caches'], DATASETS[arch]['clean_set'])

In [ ]:
# Ablation: MARGIN_S (fixed MARGIN_H=0.3)
_FIXED_MARGIN_H = 0.3
if 'ABLATION_RESULTS' not in dir(): ABLATION_RESULTS = {}

for arch in ['distilbert', 'bert']:
    if arch not in ABLATION_RESULTS: ABLATION_RESULTS[arch] = {}
    d   = DATASETS[arch]
    m   = LOADED_MODELS[arch]
    W0  = m['W0']
    print(f'\nAblation [{arch}] MARGIN_S  values={ABLATION_MARGIN_S_VALUES}  '
          f'MARGIN_H={_FIXED_MARGIN_H}')
    results = []
    for ms in ABLATION_MARGIN_S_VALUES:
        _orig_S, _orig_H = MARGIN_S, MARGIN_H
        MARGIN_S = ms
        MARGIN_H = _FIXED_MARGIN_H

        W_dict, log, elapsed = run_repair(
            arch, d['repair_set'], d['remain_set'],
            rank=RANK, max_iter=MAX_ITER,
            lambda_val=LAMBDA, reg_val=REG)

        MARGIN_S, MARGIN_H = _orig_S, _orig_H

        gaps     = np.array([gap_w(s['v'], s['gt'], W_dict) for s in d['repair_set']])
        g_sat_ms = int((gaps >= ms - GAP_TOL).sum())
        conv     = next((e['iter'] for e in log
                         if e['gap_satisfied'] == len(d['repair_set'])), None)
        gen      = {mt: eval_acc(c, W_dict) for mt, c in d['test_caches'].items()}
        gen_mean = float(np.mean(list(gen.values()))) if gen else float('nan')
        cln      = eval_acc(d['clean_set'], W_dict, use_gt=False)
        act_lip  = ACT_LIP[arch]
        sigma_W  = float(np.linalg.norm(W_dict['W'],  ord=2))
        sigma_Wc = float(np.linalg.norm(W_dict['Wc'], ord=2))
        L        = sigma_W * sigma_Wc * act_lip
        eps_star = np.where(gaps > 0, gaps / (2 * L), 0.0)

        entry = {
            'margin_s':        ms,
            'repair_acc':      eval_acc(d['repair_set'], W_dict),
            'remain_acc':      eval_acc(d['remain_set'], W_dict, use_gt=False),
            'clean_acc':       cln,
            'gen_mean':        gen_mean,
            'gen':             gen,
            'gap_sat':         g_sat_ms,
            'gap_sat_pct':     100.0 * g_sat_ms / len(d['repair_set']),
            'n_repair':        len(d['repair_set']),
            'converged_at':    conv,
            'elapsed':         elapsed,
            'dW_fro':          float(np.linalg.norm(W_dict['W'] - W0, 'fro')),
            'spec_norm':       sigma_W,
            'gap_min':         float(gaps.min()),
            'gap_mean':        float(gaps.mean()),
            'eps_star_median': float(np.median(eps_star)),
            'eps_star_mean':   float(np.mean(eps_star)),
        }
        results.append(entry)
        print(f'  MARGIN_S={ms:<4}  '
              f'repair={entry["repair_acc"]:>6.1f}%  '
              f'remain={entry["remain_acc"]:>6.1f}%  '
              f'clean={cln:>6.1f}%  '
              f'gen={gen_mean:>6.1f}%  '
              f'gap_sat={g_sat_ms}/{len(d["repair_set"])} ({entry["gap_sat_pct"]:.0f}%)  '
              f'gap_min={entry["gap_min"]:.3f}  '
              f'eps*_med={entry["eps_star_median"]:.4f}  '
              f'||dW||_F={entry["dW_fro"]:.4f}  '
              f'conv={conv}  t={elapsed:.1f}s')
    ABLATION_RESULTS[arch]['margin_s'] = results

In [ ]:
# Ablation: MARGIN_H (fixed MARGIN_S=1.0)
_FIXED_MARGIN_S = 1.0
if 'ABLATION_RESULTS' not in dir(): ABLATION_RESULTS = {}

for arch in ['distilbert', 'bert']:
    if arch not in ABLATION_RESULTS: ABLATION_RESULTS[arch] = {}
    d   = DATASETS[arch]
    m   = LOADED_MODELS[arch]
    W0  = m['W0']
    print(f'\nAblation [{arch}] MARGIN_H  values={ABLATION_MARGIN_H_VALUES}  '
          f'MARGIN_S={_FIXED_MARGIN_S}')
    results = []
    for mh in ABLATION_MARGIN_H_VALUES:
        _orig_S, _orig_H = MARGIN_S, MARGIN_H
        MARGIN_S = _FIXED_MARGIN_S
        MARGIN_H = mh

        W_dict, log, elapsed = run_repair(
            arch, d['repair_set'], d['remain_set'],
            rank=RANK, max_iter=MAX_ITER,
            lambda_val=LAMBDA, reg_val=REG)

        MARGIN_S, MARGIN_H = _orig_S, _orig_H

        gaps_rep = np.array([gap_w(s['v'], s['gt'],  W_dict) for s in d['repair_set']])
        gaps_rem = np.array([gap_w(s['v'], s['pred'], W_dict) for s in d['remain_set']])
        g_sat    = int((gaps_rep >= _FIXED_MARGIN_S - GAP_TOL).sum())
        rem_sat  = int((gaps_rem >= mh).sum())
        conv     = next((e['iter'] for e in log
                         if e['gap_satisfied'] == len(d['repair_set'])), None)
        gen      = {mt: eval_acc(c, W_dict) for mt, c in d['test_caches'].items()}
        gen_mean = float(np.mean(list(gen.values()))) if gen else float('nan')
        cln      = eval_acc(d['clean_set'], W_dict, use_gt=False)
        act_lip  = ACT_LIP[arch]
        sigma_W  = float(np.linalg.norm(W_dict['W'],  ord=2))
        sigma_Wc = float(np.linalg.norm(W_dict['Wc'], ord=2))
        L        = sigma_W * sigma_Wc * act_lip
        eps_star = np.where(gaps_rep > 0, gaps_rep / (2 * L), 0.0)

        entry = {
            'margin_h':        mh,
            'repair_acc':      eval_acc(d['repair_set'], W_dict),
            'remain_acc':      eval_acc(d['remain_set'], W_dict, use_gt=False),
            'clean_acc':       cln,
            'gen_mean':        gen_mean,
            'gen':             gen,
            'gap_sat':         g_sat,
            'gap_sat_pct':     100.0 * g_sat / len(d['repair_set']),
            'rem_gap_sat':     rem_sat,
            'rem_gap_sat_pct': 100.0 * rem_sat / len(d['remain_set']),
            'rem_gap_min':     float(gaps_rem.min()),
            'rem_gap_mean':    float(gaps_rem.mean()),
            'n_repair':        len(d['repair_set']),
            'converged_at':    conv,
            'elapsed':         elapsed,
            'dW_fro':          float(np.linalg.norm(W_dict['W'] - W0, 'fro')),
            'spec_norm':       sigma_W,
            'eps_star_median': float(np.median(eps_star)),
            'eps_star_mean':   float(np.mean(eps_star)),
        }
        results.append(entry)
        print(f'  MARGIN_H={mh:<5}  '
              f'repair={entry["repair_acc"]:>6.1f}%  '
              f'remain={entry["remain_acc"]:>6.1f}%  '
              f'clean={cln:>6.1f}%  '
              f'gen={gen_mean:>6.1f}%  '
              f'gap_sat={g_sat}/{len(d["repair_set"])} ({entry["gap_sat_pct"]:.0f}%)  '
              f'rem_gap_sat={rem_sat}/{len(d["remain_set"])} ({entry["rem_gap_sat_pct"]:.0f}%)  '
              f'rem_gap_min={entry["rem_gap_min"]:.3f}  '
              f'eps*_med={entry["eps_star_median"]:.4f}  '
              f'||dW||_F={entry["dW_fro"]:.4f}  '
              f'conv={conv}  t={elapsed:.1f}s')
    ABLATION_RESULTS[arch]['margin_h'] = results

In [ ]:
# Ablation: reg
if 'ABLATION_RESULTS' not in dir(): ABLATION_RESULTS = {}

for arch in ['distilbert', 'bert']:
    if arch not in ABLATION_RESULTS: ABLATION_RESULTS[arch] = {}
    ABLATION_RESULTS[arch]['reg'] = run_ablation(
        arch, 'reg', ABLATION_REG_VALUES,
        DATASETS[arch]['repair_set'], DATASETS[arch]['remain_set'],
        DATASETS[arch]['test_caches'], DATASETS[arch]['clean_set'])

In [ ]:
# Ablation: joint_size (repair & remain grown together)
# Each step scales repair_set and remain_set at the same time, growing roughly
# geometrically (~x2 per step); the last step is both sets at full size.
#   repair sizes:  about [10, 20, 40, 80, full_repair_n]
#   remain sizes:  N_REMAIN=800 scaled by the same fraction, ~[75, 150, 300, 600, 800]

if 'ABLATION_RESULTS' not in dir(): ABLATION_RESULTS = {}

JOINT_SIZE_ARCHS = ['distilbert', 'bert']   # <- edit to select models

def run_ablation_joint_size(arch):
    d   = DATASETS[arch]
    m   = LOADED_MODELS[arch]
    W0  = m['W0']

    full_repair_n = len(d['repair_set'])
    full_remain_n = len(d['remain_set'])   # = N_REMAIN = 800

    N_STEPS = 4   # excludes the full-size point, so N_STEPS+1 points are run
    min_repair = max(10, int(full_repair_n * 0.08))
    ratio = (full_repair_n / min_repair) ** (1.0 / N_STEPS)
    repair_sizes_raw = [int(round(min_repair * (ratio ** k))) for k in range(N_STEPS)]
    repair_sizes_raw.append(full_repair_n)
    repair_sizes = sorted(set(repair_sizes_raw))

    # remain scaled to the same fraction
    remain_sizes = []
    for r in repair_sizes:
        frac = r / full_repair_n
        rem_n = int(round(frac * full_remain_n))
        rem_n = max(1, min(rem_n, full_remain_n))
        remain_sizes.append(rem_n)

    pairs = list(zip(repair_sizes, remain_sizes))

    rng_ab  = np.random.default_rng(0)
    rep_idx = list(range(full_repair_n)); rng_ab.shuffle(rep_idx)
    rem_idx = list(range(full_remain_n)); rng_ab.shuffle(rem_idx)

    print(f'\nAblation [{arch}] joint_size  (repair, remain) pairs={pairs}')
    print(f'  full: repair={full_repair_n}  remain={full_remain_n}')
    results = []

    for (rep_n, rem_n) in pairs:
        rep = [d['repair_set'][i] for i in rep_idx[:rep_n]]
        rem = [d['remain_set'][i] for i in rem_idx[:rem_n]]

        W_dict, log, elapsed = run_repair(
            arch, rep, rem,
            rank=RANK, max_iter=MAX_ITER,
            lambda_val=LAMBDA, reg_val=REG)

        g_last   = log[-1]['gap_satisfied'] if log else 0
        conv     = next((e['iter'] for e in log
                         if e['gap_satisfied'] == len(rep)), None)
        gen      = {mt: eval_acc(c, W_dict) for mt, c in d['test_caches'].items()}
        gen_mean = float(np.mean(list(gen.values()))) if gen else float('nan')
        cln      = eval_acc(d['clean_set'], W_dict, use_gt=False)
        rem_acc  = eval_acc(rem, W_dict, use_gt=False)
        rep_acc  = eval_acc(rep, W_dict)

        is_full = (rep_n == full_repair_n and rem_n == full_remain_n)
        entry = {
            'rep_n':        rep_n,
            'rem_n':        rem_n,
            'value':        (rep_n, rem_n),   # keeps the shape other ablations use
            'is_full':      is_full,
            'repair_acc':   rep_acc,
            'remain_acc':   rem_acc,
            'clean_acc':    cln,
            'gen_mean':     gen_mean,
            'gen':          gen,
            'gap_sat':      g_last,
            'n_repair':     len(rep),
            'converged_at': conv,
            'elapsed':      elapsed,
            'dW_fro':       float(np.linalg.norm(W_dict['W'] - W0, 'fro')),
            'spec_norm':    float(np.linalg.norm(W_dict['W'], ord=2)),
        }
        results.append(entry)
        full_marker = ' [FULL]' if is_full else ''
        print(f'  repair_n={rep_n:>4d}  remain_n={rem_n:>4d}{full_marker}  '
              f'repair={rep_acc:>6.1f}%  remain={rem_acc:>6.1f}%  '
              f'clean={cln:>6.1f}%  gen={gen_mean:>6.1f}%  '
              f'gap={g_last}/{len(rep)}  conv={conv}  t={elapsed:.1f}s')

    return results


for arch in JOINT_SIZE_ARCHS:
    if arch not in ABLATION_RESULTS: ABLATION_RESULTS[arch] = {}
    ABLATION_RESULTS[arch]['joint_size'] = run_ablation_joint_size(arch)

## Baselines — LoRA / Full FT

The baseline datasets are rebuilt from raw text using the **original** (pre-FT) model's predictions, independently of the WARP caches:

- `repair` — AdvGLUE dev samples the model gets **wrong**
- `remain` — `N_REMAIN` Yelp samples the model gets **right**
- `clean` — `N_CLEAN_TARGET` Yelp samples the model gets **right**, non-overlapping with `remain`
- `test_by_method` — the attack sets, used for generalization evaluation

Every entry is a `{'text': str, 'label': int}` dict.

**Training data**: `repair` + `remain`, both with ground-truth labels.
**Eval**: plain PyTorch inference.


In [ ]:
# Build raw-text datasets from the original model's predictions

RAW_DATA = {}  # arch -> {repair, remain, clean, test_by_method}


def predict_all(model, tok, texts, batch_size=64):
    model.eval()
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i: i+batch_size], return_tensors='pt',
                  truncation=True, max_length=128, padding=True)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            preds.extend(model(**enc).logits.argmax(1).cpu().tolist())
    return preds


def build_raw_data(arch):
    cfg      = MODEL_CONFIGS[arch]
    tok      = AutoTokenizer.from_pretrained(cfg['model_id'])
    model    = AutoModelForSequenceClassification.from_pretrained(
                   cfg['model_id']).to(DEVICE)
    model.eval()

    # dev -> repair set
    with open(DEV_PATH) as f: dev_ann = json.load(f)
    dev_items  = dev_ann['sst2']
    dev_texts  = [x['sentence'] for x in dev_items]
    dev_labels = [int(x['label']) for x in dev_items]
    dev_preds  = predict_all(model, tok, dev_texts)
    repair = [
        {'text': t, 'label': l}
        for t, l, p in zip(dev_texts, dev_labels, dev_preds) if p != l
    ]
    print(f'[{arch}] repair={len(repair)}')

    # Yelp -> remain + clean
    yelp = load_dataset('fancyzhx/yelp_polarity', split='test')
    pool = [{'text': ex['text'][:256], 'label': int(ex['label'])} for ex in yelp]
    np.random.default_rng(SEED).shuffle(pool)

    pool_rem   = pool[:N_REMAIN * 2]
    rem_preds  = predict_all(model, tok, [x['text'] for x in pool_rem])
    remain     = [
        {'text': x['text'], 'label': x['label']}
        for x, p in zip(pool_rem, rem_preds) if p == x['label']
    ][:N_REMAIN]
    print(f'[{arch}] remain={len(remain)}')

    pool_cln   = pool[N_REMAIN*2 : N_REMAIN*2 + N_CLEAN_POOL]
    cln_preds  = predict_all(model, tok, [x['text'] for x in pool_cln])
    clean      = [
        {'text': x['text'], 'label': x['label']}
        for x, p in zip(pool_cln, cln_preds) if p == x['label']
    ][:N_CLEAN_TARGET]
    print(f'[{arch}] clean={len(clean)}')

    # attack test sets
    with open(TEST_PATH) as f: test_ann = json.load(f)
    by_method = defaultdict(list)
    for x in test_ann['sst2']:
        if x.get('method') in EXCLUDED_METHODS or x.get('method') == 'glue':
            continue
        by_method[x['method']].append({'text': x['sentence'], 'label': int(x['label'])})
    print(f'[{arch}] attacks={sorted(by_method.keys())}')

    del model; torch.cuda.empty_cache()
    RAW_DATA[arch] = {
        'repair': repair, 'remain': remain,
        'clean': clean, 'test_by_method': dict(by_method),
    }


print('Dataset builder defined.')

In [ ]:
# PyTorch eval + Dataset

class TextDataset(Dataset):
    def __init__(self, items, tok, max_length=128):
        self.items, self.tok, self.max_length = items, tok, max_length

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        enc  = self.tok(item['text'], truncation=True, max_length=self.max_length,
                        padding='max_length', return_tensors='pt')
        out  = {k: v.squeeze(0) for k, v in enc.items()}
        out['label'] = torch.tensor(item['label'], dtype=torch.long)
        return out


def eval_items(model, tok, items, batch_size=32):
    if not items: return 0.0
    loader = DataLoader(TextDataset(items, tok), batch_size=batch_size)
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            lbs    = batch.pop('label').to(DEVICE)
            kw     = {k: v.to(DEVICE) for k, v in batch.items()}
            preds  = model(**kw).logits.argmax(1)
            correct += (preds == lbs).sum().item()
            total   += lbs.size(0)
    return 100.0 * correct / total


def baseline_eval(model, tok, arch, data, label=''):
    tag = label or arch
    print(f'\n{"="*65}')
    print(f'EVAL  {tag}')
    print(f'{"="*65}')

    r_acc    = eval_items(model, tok, data['repair'])
    rem_acc  = eval_items(model, tok, data['remain'])
    cln_acc  = eval_items(model, tok, data['clean'])
    gen      = {mt: eval_items(model, tok, items)
                for mt, items in data['test_by_method'].items()}
    gen_mean = float(np.mean(list(gen.values()))) if gen else float('nan')

    print(f'  Repair acc:  {r_acc:.1f}%')
    print(f'  Remain acc:  {rem_acc:.1f}%')
    print(f'  Clean acc:   {cln_acc:.1f}%  (n={len(data["clean"])})')
    print(f'  Gen mean:    {gen_mean:.1f}%')
    for mt in sorted(gen): print(f'    {mt:<22} {gen[mt]:.1f}%')
    print(f'[FACT 1/2/3/4] N/A  (no certificate from FT)')
    print(f'{"─"*65}')

    return {'arch': arch, 'label': tag,
            'repair_acc': r_acc, 'remain_acc': rem_acc, 'clean_acc': cln_acc,
            'gen': gen, 'gen_mean': gen_mean}


print('Eval utilities defined.')

In [ ]:
# Training loop

def run_ft(model, tok, repair, remain, trainable_params, label='',
           lr=BL_LR, epochs=BL_EPOCHS, batch_size=BL_BATCH_SIZE, patience=BL_PATIENCE):
    print(f'\n--- FT: {label}  repair={len(repair)}  remain={len(remain)} ---')
    train_items = repair + remain
    loader      = DataLoader(TextDataset(train_items, tok),
                             batch_size=batch_size, shuffle=True)
    rep_loader  = DataLoader(TextDataset(repair, tok), batch_size=batch_size)
    rem_loader  = DataLoader(TextDataset(remain, tok), batch_size=batch_size)

    optim = torch.optim.AdamW(trainable_params, lr=lr)
    ce    = nn.CrossEntropyLoss()

    def loader_acc(dl):
        model.eval()
        c = t = 0
        with torch.no_grad():
            for batch in dl:
                lbs   = batch.pop('label').to(DEVICE)
                kw    = {k: v.to(DEVICE) for k, v in batch.items()}
                c    += (model(**kw).logits.argmax(1) == lbs).sum().item()
                t    += lbs.size(0)
        return 100.0 * c / t if t else 0.0

    best_acc   = -1.0
    best_loss  = float('inf')
    best_state = copy.deepcopy(model.state_dict())
    wait       = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in loader:
            lbs  = batch.pop('label').to(DEVICE)
            kw   = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = ce(model(**kw).logits, lbs)
            optim.zero_grad(); loss.backward(); optim.step()
            total_loss += loss.item()

        r = loader_acc(rep_loader)
        m = loader_acc(rem_loader)
        print(f'  ep={ep:3d}  loss={total_loss/len(loader):.4f}  '
              f'repair={r:.1f}%  remain={m:.1f}%')

        if r > best_acc:
            best_acc = r; best_state = copy.deepcopy(model.state_dict())

        curr_loss = total_loss / len(loader)
        if curr_loss < best_loss - 1e-6:
            best_loss = curr_loss; wait = 0
        else:
            wait += 1

        if wait >= patience:
            print(f'  Early stop (loss_wait={wait})')
            break

    model.load_state_dict(best_state)
    model.eval()
    print(f'  Best repair={best_acc:.1f}%')
    return model


print('Training loop defined.')

In [ ]:
# LoRA

class LoRALinear(nn.Module):
    def __init__(self, linear: nn.Linear, rank: int, alpha: float):
        super().__init__()
        d_out, d_in    = linear.weight.shape
        self.linear    = linear
        self.scale     = alpha / rank
        self.lora_A    = nn.Parameter(torch.randn(rank, d_in,  device=DEVICE) * 0.01)
        self.lora_B    = nn.Parameter(torch.zeros(d_out, rank, device=DEVICE))
        for p in self.linear.parameters(): p.requires_grad = False

    def forward(self, x):
        return self.linear(x) + self.scale * (x @ self.lora_A.T @ self.lora_B.T)


def inject_lora(model, arch, rank, alpha):
    for p in model.parameters(): p.requires_grad = False
    target = MODEL_CONFIGS[arch]['lora_target']
    parts  = target.split('.')
    parent = model
    for part in parts[:-1]: parent = getattr(parent, part)
    orig   = getattr(parent, parts[-1])
    lora   = LoRALinear(orig, rank=rank, alpha=alpha).to(DEVICE)
    setattr(parent, parts[-1], lora)
    print(f'  LoRA -> {target}  shape={tuple(orig.weight.shape)}  rank={rank}')
    return lora


print('LoRA defined.')

### Baselines — build datasets


In [ ]:
for arch in ['distilbert', 'bert', 'roberta']:
    build_raw_data(arch)

In [ ]:
# Attack gen set sizes
print('Attack gen set sizes (test_by_method):')
print()
for arch in ['distilbert', 'bert']:
    if arch not in RAW_DATA:
        print(f'[{arch}] RAW_DATA not built yet — run Phase 1 first.')
        continue
    tbm = RAW_DATA[arch]['test_by_method']
    total = sum(len(v) for v in tbm.values())
    print(f'[{arch}]  ({len(tbm)} attack methods, {total} examples total)')
    for method in sorted(tbm):
        print(f'  {method:<30}  n = {len(tbm[method])}')
    print()

In [ ]:
BL_RESULTS = {}


### Baselines — same-layer LoRA (rank=2)


In [ ]:
for arch in ['distilbert', 'bert', 'roberta']:
    cfg   = MODEL_CONFIGS[arch]
    tok   = AutoTokenizer.from_pretrained(cfg['model_id'])
    data  = RAW_DATA[arch]
    model = AutoModelForSequenceClassification.from_pretrained(
                cfg['model_id']).to(DEVICE)

    lora      = inject_lora(model, arch, rank=BL_LORA_RANK, alpha=BL_LORA_ALPHA)
    trainable = [lora.lora_A, lora.lora_B]
    print(f'[{arch}] LoRA trainable: {sum(p.numel() for p in trainable):,}')

    baseline_eval(model, tok, arch, data,
                  label=f'{arch} LoRA r={BL_LORA_RANK} (before)')
    t0 = time.time()
    run_ft(model, tok, data['repair'], data['remain'], trainable,
           label=f'{arch} same-layer LoRA r={BL_LORA_RANK}')
    elapsed = time.time() - t0
    res = baseline_eval(model, tok, arch, data,
                        label=f'{arch} same-layer LoRA r={BL_LORA_RANK} (after, {elapsed:.0f}s)')
    BL_RESULTS.setdefault(arch, {})['same_layer_lora'] = res
    del model; torch.cuda.empty_cache()

### Baselines — all-layers full FT


In [ ]:
for arch in ['distilbert', 'bert', 'roberta']:
    cfg   = MODEL_CONFIGS[arch]
    tok   = AutoTokenizer.from_pretrained(cfg['model_id'])
    data  = RAW_DATA[arch]
    model = AutoModelForSequenceClassification.from_pretrained(
                cfg['model_id']).to(DEVICE)

    for p in model.parameters(): p.requires_grad = True
    trainable = list(model.parameters())
    print(f'[{arch}] all-layers trainable: {sum(p.numel() for p in trainable):,}')

    baseline_eval(model, tok, arch, data,
                  label=f'{arch} all-layers FT (before)')
    t0 = time.time()
    run_ft(model, tok, data['repair'], data['remain'], trainable,
           label=f'{arch} all-layers full FT')
    elapsed = time.time() - t0
    res = baseline_eval(model, tok, arch, data,
                        label=f'{arch} all-layers full FT (after, {elapsed:.0f}s)')
    BL_RESULTS.setdefault(arch, {})['all_layers_full_ft'] = res
    del model; torch.cuda.empty_cache()

### Baselines — summary


In [ ]:
BASELINE_NAMES = {
    'same_layer_full_ft': 'Same-layer Full FT',
    'same_layer_lora':    f'Same-layer LoRA (r={BL_LORA_RANK})',
    'all_layers_full_ft': 'All-layers Full FT',
}
W = 90
print('=' * W)
print(f'{"Model":<12}  {"Method":<26}  {"Repair":>7}  {"Remain":>7}  '
      f'{"Clean":>7}  {"Gen mean":>9}  {"Provable?":>9}')
print('-' * W)
for arch in ['distilbert', 'bert', 'roberta']:
    for key, name in BASELINE_NAMES.items():
        if arch not in BL_RESULTS or key not in BL_RESULTS[arch]: continue
        r = BL_RESULTS[arch][key]
        print(f'{arch:<12}  {name:<26}  {r["repair_acc"]:>6.1f}%  '
              f'{r["remain_acc"]:>6.1f}%  {r["clean_acc"]:>6.1f}%  '
              f'{r["gen_mean"]:>8.1f}%  {"No":>9}')
print('=' * W)

In [ ]:
save_path = f'{BASELINE_RESULT_DIR}/baseline_results.pkl'
with open(save_path, 'wb') as f: pickle.dump(BL_RESULTS, f)
print(f'Saved: {save_path}')